# MIPDS Milestone 6 — Vision Module Freeze and Interface Specification

**Aaron Tobias · CSC 5352 Deep Learning**

The frozen ResNet-50 achieved **60.00% held-out top-1 accuracy** on 570 side-view frames from four participants. This notebook includes the executed evaluation, visual analyses, model card, and interface specification for MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d.


## Run record

The saved Colab audit verified the checkpoint, split manifest, and test frame bytes before evaluating the test set. The written deliverables use that run's saved outputs and figures. The training date of 2026-10-11 comes from the checkpoint saved at the end of the M4 fine-tuning cell.

## Reproduce the audit

1. Set `PROJECT_DIR` to the folder containing the original `classification_manifest.csv`, `milestone4_outputs/m4_run_record.json`, `milestone4_outputs/m4_finetuned.keras`, and the held-out frame paths under `frames/`. Colab mounts Drive automatically for a project path under `/content/drive`.
2. Run all cells in sequence. The notebook saves the metrics, figures, and reports under the results folder printed in the configuration cell.
3. The written analyses reference the exact frames and comparisons displayed here. Check those observations against the new figures if a future run changes the displayed cases.


In [1]:
import importlib.util
import subprocess
import sys
packages = {
    "tensorflow": "tensorflow>=2.16,<3",
    "numpy": "numpy", "pandas": "pandas", "PIL": "pillow",
    "jinja2": "Jinja2",
    "matplotlib": "matplotlib", "sklearn": "scikit-learn>=1.4,<2",
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Audit dependencies available. Prefer the same framework versions as the M4 training run.")

Audit dependencies available. Prefer the same framework versions as the M4 training run.


In [2]:
from pathlib import Path
from datetime import date, datetime, timezone
from collections import Counter
import hashlib
import importlib.metadata
import inspect
import json
import math
import pprint
import re
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from IPython.display import Markdown, display
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
VERSIONS = {name: importlib.metadata.version(name) for name in
            ["tensorflow", "keras", "scikit-learn", "numpy", "pandas", "pillow"]}
display(pd.Series(VERSIONS, name="audit_runtime_version"))

,audit_runtime_version
tensorflow,2.21.0
keras,3.13.2
scikit-learn,1.6.1
numpy,2.4.6
pandas,2.2.3
pillow,11.3.0


In [3]:
# Set this to the folder containing your MIPDS experiment files.
PROJECT_DIR = "/content/drive/MyDrive/MIPDS/mipds_public_exercises"
ROOT = Path(PROJECT_DIR).expanduser()
DRIVE_MOUNT = Path("/content/drive")
if ROOT.is_relative_to(DRIVE_MOUNT):
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
ROOT = ROOT.resolve()
TRAINING_COMPLETED_DATE = "2026-10-11"  # The M4 fine-tuning cell saved this checkpoint at the end of training.
BATCH_SIZE = 8
N_BINS = 10
ERROR_SAMPLE_SIZE = 30
GRADCAM_COUNT = 6
NN_QUERY_COUNT = 5
MAX_EMBEDDING_MEMORY_MIB = 1024
if TRAINING_COMPLETED_DATE:
    date.fromisoformat(TRAINING_COMPLETED_DATE)
M4_RECORD_PATH = ROOT / "milestone4_outputs" / "m4_run_record.json"
MANIFEST_PATH = ROOT / "classification_manifest.csv"
if not M4_RECORD_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {M4_RECORD_PATH}. Run the public-exercise M4 notebook through its "
        "export cell and restore the same complete project folder. M6 cannot invent a checkpoint.")
m4 = json.loads(M4_RECORD_PATH.read_text(encoding="utf-8"))
required = {"milestone", "task", "dataset", "backbone", "pretrained_weights",
            "nested_backbone", "feature_layer", "input_size", "preprocessing",
            "preprocessing_location", "resize_method", "classes", "evaluation_scope",
            "checkpoint_path", "checkpoint_sha256", "classification_manifest_sha256",
            "stages", "frozen_weight_checks", "split_sizes", "audit"}
if not required.issubset(m4):
    raise ValueError(f"M4 export is incomplete: {sorted(required - set(m4))}.")
if (m4["milestone"] != 4 or m4["backbone"] != "resnet50"
    or m4["pretrained_weights"] != "imagenet"
    or m4["task"].get("name") != "exercise_recognition"
    or m4["preprocessing"] != "resnet50"
    or m4["preprocessing_location"] != "outside_saved_model"
    or m4["input_size"] != [224, 224]
    or m4["resize_method"] != "bilinear"
    or m4["feature_layer"] != "conv5_block3_out"
    or m4["evaluation_scope"] != "unseen_people"):
    raise ValueError("This adapter is for the latest public-exercise ResNet50 M4 export. "
                     "Do not substitute the synthetic Chapter 4 or Chapter 6 models.")
EXPECTED_CLASSES = ["dumbbell_side_bend", "bent_over_row", "bicep_curl",
                    "push_up", "shoulder_press", "squat", "triceps_extension"]
if m4["classes"] != EXPECTED_CLASSES:
    raise ValueError("Unexpected label order. Adapt the contract explicitly rather than relabel outputs.")
if (m4["dataset"].get("doi") != "10.17632/kgbb3yn47p.3"
    or m4["dataset"].get("camera_view") != "side"):
    raise ValueError("Expected the recorded Version 3 side-view public-exercise dataset.")
if len(m4["stages"]) != 2 or any(
    stage.get("epochs_completed", 0) < minimum
    for stage, minimum in zip(m4["stages"], [10, 5])
):
    raise ValueError("Require completed M4 head training (10+ epochs) and fine-tuning (5+ epochs).")
if not all(m4["frozen_weight_checks"].get(k) is True
           for k in ["stage1", "stage2", "stage2_unfrozen_changed"]):
    raise ValueError("M4 freezing/fine-tuning checks did not pass.")
CLASSES = m4["classes"]
INPUT_SIZE = tuple(m4["input_size"])
CHECKPOINT = (ROOT / m4["checkpoint_path"]).resolve()
if not CHECKPOINT.is_relative_to(ROOT):
    raise ValueError("Checkpoint path must stay inside the project folder.")
VERSION_ID = "MIPDS-Vision-ResNet50-exercise-v1-" + m4["checkpoint_sha256"][:12]
OUTPUT = ROOT / "milestone6_outputs" / m4["checkpoint_sha256"][:12]
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Audit:", VERSION_ID, "\nProject folder:", ROOT, "\nResults folder:", OUTPUT)

Mounted at /content/drive
Audit: MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d 
Project folder: /content/drive/MyDrive/MIPDS/mipds_public_exercises 
Results folder: /content/drive/MyDrive/MIPDS/mipds_public_exercises/milestone6_outputs/0d59d2a0080d


In [4]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_path(value):
    path = Path(value)
    resolved = (ROOT / path).resolve()
    if not resolved.is_relative_to(ROOT):
        raise ValueError(f"Frame path is outside the project folder: {value}")
    return resolved

def validate_manifest(table, record):
    required = ["frame_id", "path", "label", "split", "client_id",
                "session_id", "video_path", "sha256", "source_form_label", "view"]
    if not set(required).issubset(table.columns):
        raise ValueError("Restore the complete classification manifest exported by M4.")
    if table.empty or table[required].apply(lambda c: c.str.strip().eq("")).any().any():
        raise ValueError("The classification manifest is empty or has blank required values.")
    if table.frame_id.duplicated().any() or table.path.duplicated().any():
        raise ValueError("Duplicate frame IDs or frame paths.")
    if set(table.split) != {"train", "validation", "test"}:
        raise ValueError("Require the original train, validation, and held-out test splits.")
    if set(table.label) != set(record["classes"]) or not table.view.eq("side").all():
        raise ValueError("Labels or camera views differ from the recorded task.")
    for key in ["client_id", "session_id", "video_path", "sha256"]:
        if table.groupby(key).split.nunique().gt(1).any():
            raise ValueError(f"Data leakage: {key} crosses splits.")
    if table.split.value_counts().to_dict() != record["split_sizes"]:
        raise ValueError("Split sizes differ from the training record.")
    for split in ["train", "validation", "test"]:
        if set(table.loc[table.split.eq(split), "label"]) != set(record["classes"]):
            raise ValueError(f"M4 {split} lacks a recorded class.")
    return table

def verify_test_frames(test):
    result = test.copy()
    result["resolved_path"] = result.path.map(resolve_path)
    for row in result.itertuples(index=False):
        if not row.resolved_path.is_file():
            raise FileNotFoundError(f"Restore held-out frame: {row.path}")
        if sha256_file(row.resolved_path) != row.sha256:
            raise ValueError(f"Held-out frame contents differ from the M4 manifest: {row.frame_id}.")
    return result

def summarize_predictions(probabilities, true_indices, classes, n_bins=10):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    true_indices = np.asarray(true_indices)
    if probabilities.ndim != 2 or probabilities.shape != (len(true_indices), len(classes)):
        raise ValueError("Prediction dimensions do not match labels and recorded classes.")
    if len(true_indices) == 0 or true_indices.dtype.kind not in "iu":
        raise ValueError("Require a nonempty integer target vector.")
    if (true_indices < 0).any() or (true_indices >= len(classes)).any():
        raise ValueError("Target index is outside the recorded class order.")
    if (not np.isfinite(probabilities).all() or (probabilities < 0).any()
        or (probabilities > 1).any()
        or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)):
        raise ValueError("Require finite softmax probability vectors, not logits.")
    if n_bins < 1:
        raise ValueError("Need a positive bin count.")
    predicted = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    correct = predicted == true_indices
    cm = confusion_matrix(true_indices, predicted, labels=np.arange(len(classes)))
    support = cm.sum(axis=1)
    per_class = pd.DataFrame({
        "class": classes, "test_frames": support, "correct": cm.diagonal(),
        "accuracy": np.divide(cm.diagonal(), support,
                              out=np.full(len(classes), np.nan), where=support > 0),
    })
    # Confidence exactly 1.0 belongs to the final bin; empty bins remain visible as NaN.
    bin_index = np.minimum((confidence * n_bins).astype(int), n_bins - 1)
    bin_records = []
    for index in range(n_bins):
        mask = bin_index == index
        bin_records.append({
            "lower": index / n_bins, "upper": (index + 1) / n_bins,
            "count": int(mask.sum()),
            "mean_confidence": float(confidence[mask].mean()) if mask.any() else np.nan,
            "accuracy": float(correct[mask].mean()) if mask.any() else np.nan,
        })
    bins = pd.DataFrame(bin_records)
    populated = bins.loc[bins["count"].gt(0)]
    ece = float(np.sum(populated["count"] / len(correct) *
                      np.abs(populated.accuracy - populated.mean_confidence)))
    summary = {
        "n_test": len(correct), "top1_accuracy": float(correct.mean()),
        "mean_confidence": float(confidence.mean()),
        "signed_confidence_gap": float(confidence.mean() - correct.mean()),
        "ece": ece, "n_bins": n_bins,
        "confidence_min": float(confidence.min()), "confidence_max": float(confidence.max()),
        "test_errors": int((~correct).sum()),
    }
    return predicted, confidence, correct, cm, per_class, bins, summary

def select_error_indices(correct, seed=42, requested=30):
    if requested < 1:
        raise ValueError("Request at least one error.")
    errors = np.flatnonzero(~np.asarray(correct, dtype=bool))
    count = min(requested, len(errors))
    return np.random.default_rng(seed).choice(errors, count, replace=False)

def markdown_table(table, float_columns=()):
    'Small Markdown renderer without an optional tabulate dependency.'
    def cell(value, column):
        if pd.isna(value):
            return "Not evaluated"
        if column in float_columns:
            return f"{float(value):.2%}"
        return str(value).replace("|", "/").replace("\n", " ")
    header = "| " + " | ".join(map(str, table.columns)) + " |"
    rule = "| " + " | ".join(["---"] * len(table.columns)) + " |"
    rows = ["| " + " | ".join(cell(v, col) for col, v in zip(table.columns, row)) + " |"
            for row in table.itertuples(index=False, name=None)]
    return "\n".join([header, rule, *rows])

def sentence_count(text):
    return len([s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()])

print("Audit helpers loaded.")

Audit helpers loaded.


In [5]:
if not CHECKPOINT.is_file() or not MANIFEST_PATH.is_file():
    raise FileNotFoundError("Restore the trained .keras checkpoint and original classification manifest.")
if sha256_file(CHECKPOINT) != m4["checkpoint_sha256"]:
    raise ValueError("Checkpoint hash differs from the M4 export.")
if sha256_file(MANIFEST_PATH) != m4["classification_manifest_sha256"]:
    raise ValueError("The manifest changed after training; do not silently redefine the test set.")
manifest = validate_manifest(pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False), m4)
test_rows = manifest.loc[manifest.split.eq("test")].sort_values("frame_id").reset_index(drop=True)
test = verify_test_frames(test_rows)
if len(test) < 3:
    raise ValueError("Need at least three real held-out frames for the representation audit.")
split_summary = manifest.groupby("split").agg(
    frames=("frame_id", "size"), participants=("client_id", "nunique"),
    source_clips=("video_path", "nunique"))
split_distribution = pd.crosstab(manifest.label, manifest.split).reindex(CLASSES, fill_value=0)
display(split_summary)
display(split_distribution)
split_distribution.to_csv(OUTPUT / "class_distribution.csv")
with zipfile.ZipFile(CHECKPOINT) as archive:
    checkpoint_metadata = json.loads(archive.read("metadata.json"))
CHECKPOINT_SAVED_AT = checkpoint_metadata.get("date_saved", "not recorded")
print("Checkpoint save timestamp:", CHECKPOINT_SAVED_AT)
print("Checkpoint Keras version:", checkpoint_metadata.get("keras_version", "not recorded"))
if checkpoint_metadata.get("keras_version") != VERSIONS["keras"]:
    print("Version difference: verify loading and reconstructed-head agreement below.")
M5_RECORD_PATH = ROOT / "milestone5_outputs" / "run_record.json"
m5 = None
if M5_RECORD_PATH.is_file():
    m5 = json.loads(M5_RECORD_PATH.read_text(encoding="utf-8"))
    if (m5.get("milestone") != 5 or m5.get("checkpoint_sha256") != m4["checkpoint_sha256"]
        or m5.get("milestone4_record_sha256") != sha256_file(M4_RECORD_PATH)):
        raise ValueError("M5 results reference a different M4 checkpoint or run record.")
    print("Matching M5 run record found; its detector is separate from this classification contract.")
else:
    print("M5 run record absent. This audit does not establish completion of Milestone 5.")
print("Verified checkpoint, manifest, and held-out image hashes. Split checks cover all rows.")

,frames,participants,source_clips
split,,,
test,570,4,48
train,2531,17,213
validation,559,4,47


split,test,train,validation
label,,,
dumbbell_side_bend,96,406,96
bent_over_row,72,396,96
bicep_curl,72,336,72
push_up,60,357,81
shoulder_press,90,348,48
squat,84,324,94
triceps_extension,96,364,72


Checkpoint save timestamp: 2026-10-11@23:41:45
Checkpoint Keras version: 3.13.2
Matching M5 run record found; its detector is separate from this classification contract.
Verified checkpoint, manifest, and held-out image hashes. Split checks cover all rows.


In [6]:
model = tf.keras.models.load_model(str(CHECKPOINT), compile=False)
model.trainable = False
backbone = model.get_layer(m4["nested_backbone"])
feature_model = tf.keras.Model(backbone.inputs, backbone.get_layer(m4["feature_layer"]).output)
gap_layer = model.get_layer("global_average_pool")
class_head = model.get_layer("exercise_class")
model.get_layer("head_dropout")  # Inactive at inference; included in the original model.
if (tuple(model.input_shape[1:]) != (224, 224, 3)
    or tuple(model.output_shape[1:]) != (len(CLASSES),)
    or tuple(feature_model.output_shape[1:]) != (7, 7, 2048)
    or not isinstance(gap_layer, tf.keras.layers.GlobalAveragePooling2D)
    or not isinstance(class_head, tf.keras.layers.Dense)
    or tuple(class_head.kernel.shape) != (2048, len(CLASSES))):
    raise ValueError("Loaded architecture does not match the frozen interface.")
if model.trainable_weights:
    raise AssertionError("The audit must not expose trainable model weights.")
PARAMETER_COUNT = int(model.count_params())
def hash_model_weights():
    digest = hashlib.sha256()
    for weight in model.weights:
        array = np.asarray(weight.numpy())
        digest.update(str(array.shape).encode())
        digest.update(str(array.dtype).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()
WEIGHTS_BEFORE_AUDIT = hash_model_weights()
def logits_from_maps(maps):
    vector = gap_layer(maps)
    return tf.linalg.matmul(vector, class_head.kernel) + class_head.bias
print("Frozen model parameters:", f"{PARAMETER_COUNT:,}")
print("Feature vector: 2048 · spatial map: 7×7×2048 NHWC · seven-class softmax.")

Frozen model parameters: 23,602,055
Feature vector: 2048 · spatial map: 7×7×2048 NHWC · seven-class softmax.


In [7]:
def preprocess_rgb(rgb):
    rgb = np.asarray(rgb)
    if (rgb.ndim != 3 or rgb.shape[2] != 3 or min(rgb.shape[:2]) < 1
        or rgb.dtype.kind not in "uif" or not np.isfinite(rgb).all()
        or np.min(rgb) < 0 or np.max(rgb) > 255):
        raise ValueError("Require a finite nonempty H×W×3 RGB array with values in [0,255].")
    resized = tf.image.resize(tf.cast(rgb, tf.float32), INPUT_SIZE, method=m4["resize_method"])
    return tf.keras.applications.resnet50.preprocess_input(resized[None, ...])

def load_rgb(path):
    # Match M4: stored PNG pixel orientation, three RGB channels, no EXIF rotation.
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))

def load_batch(paths):
    images = []
    for path in paths:
        rgb = tf.io.decode_png(tf.io.read_file(str(path)), channels=3)
        images.append(tf.image.resize(tf.cast(rgb, tf.float32), INPUT_SIZE,
                                      method=m4["resize_method"]))
    return tf.keras.applications.resnet50.preprocess_input(tf.stack(images))

def infer_vision(rgb_image):
    'Frozen Python interface; arrays are float32 and retain the recorded class/feature order.'
    tensor = preprocess_rgb(rgb_image)
    probabilities = model(tensor, training=False).numpy()[0]
    maps = feature_model(tensor, training=False).numpy()[0].astype(np.float32)
    index = int(probabilities.argmax())
    threshold = float(m4["audit"]["low_confidence_threshold"])
    return {
        "class_label": CLASSES[index],
        "confidence": float(probabilities[index]),
        "class_probabilities": {label: float(probabilities[i]) for i, label in enumerate(CLASSES)},
        "feature_vector": maps.mean(axis=(0, 1)).astype(np.float32),
        "spatial_features": maps,
        "review_required": bool(probabilities[index] < threshold),
        "model_version": VERSION_ID,
    }
print("Inference adapter defined. The review flag's inherited threshold is not a calibrated safety rule.")

Inference adapter defined. The review flag's inherited threshold is not a calibrated safety rule.


## Part 1 — Held-out test evaluation
Evaluate every original test frame exactly once per audit run with deterministic resize and inference mode. Frame-level accuracy is the required metric; repeated frames from one clip remain correlated, so frame counts are not independent-person sample counts. Calibration bins measure top-label confidence, and ECE is a descriptive statistic rather than a fitted correction. The test set is not used to choose a checkpoint, preprocessing rule, threshold, or calibrator.

In [8]:
projected_mib = len(test) * (2048 + len(CLASSES)) * 4 / 1024**2
if projected_mib > MAX_EMBEDDING_MEMORY_MIB:
    raise MemoryError("Test embedding storage exceeds the configured runtime budget.")
probability_batches, embedding_batches = [], []
for start in range(0, len(test), BATCH_SIZE):
    rows = test.iloc[start:start + BATCH_SIZE]
    images = load_batch(rows.resolved_path.tolist())
    maps = feature_model(images, training=False)
    probabilities = model(images, training=False).numpy()
    reconstructed = tf.nn.softmax(logits_from_maps(maps), axis=-1).numpy()
    if not np.allclose(probabilities, reconstructed, rtol=1e-4, atol=1e-5):
        raise ValueError("Spatial-map head reconstruction differs from the loaded classifier; fix GradCAM wiring.")
    probability_batches.append(probabilities)
    embedding_batches.append(gap_layer(maps).numpy())
probabilities = np.concatenate(probability_batches).astype(np.float32)
embeddings = np.concatenate(embedding_batches).astype(np.float32)
label_to_index = {label: i for i, label in enumerate(CLASSES)}
true_indices = test.label.map(label_to_index).to_numpy(dtype=int)
predicted, confidences, correct, cm, per_class, bins, metrics = summarize_predictions(
    probabilities, true_indices, CLASSES, N_BINS)
metrics["balanced_accuracy"] = float(balanced_accuracy_score(true_indices, predicted))
metrics["macro_f1"] = float(f1_score(true_indices, predicted, labels=range(len(CLASSES)),
                                     average="macro", zero_division=0))
predictions = test.drop(columns="resolved_path").copy()
predictions["predicted_class"] = [CLASSES[i] for i in predicted]
predictions["confidence"] = confidences
predictions["correct"] = correct
for index, label in enumerate(CLASSES):
    predictions["probability_" + label] = probabilities[:, index]
predictions.to_csv(OUTPUT / "test_predictions.csv", index=False)
per_class.to_csv(OUTPUT / "per_class_accuracy.csv", index=False)
bins.to_csv(OUTPUT / "calibration_bins.csv", index=False)
(OUTPUT / "test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
np.savez_compressed(OUTPUT / "test_embeddings.npz", frame_ids=test.frame_id.to_numpy(dtype=str),
                    embeddings=embeddings, true_indices=true_indices)
print(f"REAL held-out top-1 accuracy: {metrics['top1_accuracy']:.2%} "
      f"({int(correct.sum())}/{len(test)} frames; {test.client_id.nunique()} held-out participants)")
display(per_class.style.format({"accuracy": "{:.2%}"}, na_rep="Not evaluated"))
print(f"ECE: {metrics['ece']:.2%}; signed mean-confidence gap: "
      f"{100 * metrics['signed_confidence_gap']:+.2f} percentage points.")
print(f"Observed confidence: [{metrics['confidence_min']:.6f}, {metrics['confidence_max']:.6f}]")

REAL held-out top-1 accuracy: 60.00% (342/570 frames; 4 held-out participants)


,class,test_frames,correct,accuracy
0,dumbbell_side_bend,96,83,86.46%
1,bent_over_row,72,66,91.67%
2,bicep_curl,72,15,20.83%
3,push_up,60,36,60.00%
4,shoulder_press,90,74,82.22%
5,squat,84,25,29.76%
6,triceps_extension,96,43,44.79%


ECE: 28.28%; signed mean-confidence gap: +28.28 percentage points.
Observed confidence: [0.237117, 1.000000]


In [9]:
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(cm, cmap="Blues")
short_labels = [label.replace("_", " ") for label in CLASSES]
ax.set_xticks(range(len(CLASSES)), short_labels, rotation=45, ha="right")
ax.set_yticks(range(len(CLASSES)), short_labels)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xlabel="Predicted exercise", ylabel="True source exercise", title="Held-out test confusion matrix")
fig.colorbar(image, ax=ax, label="Frames")
fig.tight_layout()
fig.savefig(OUTPUT / "confusion_matrix.png", dpi=160)
plt.show()

populated = bins.loc[bins["count"].gt(0)]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].plot([0, 1], [0, 1], "k--", label="Perfect calibration")
axes[0].scatter(populated.mean_confidence, populated.accuracy,
                s=35 + 250 * populated["count"] / populated["count"].max(),
                alpha=0.8, label="Observed; size reflects support")
axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean top-label confidence",
            ylabel="Observed top-1 accuracy", title=f"Reliability diagram · ECE {metrics['ece']:.2%}")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)
axes[1].bar((bins.lower + bins.upper) / 2, bins["count"], width=0.9 / N_BINS)
axes[1].set(xlim=(0, 1), xlabel="Top-label confidence", ylabel="Frames", title="Bin support")
fig.tight_layout()
fig.savefig(OUTPUT / "reliability_diagram.png", dpi=160)
plt.show()
display(bins)

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

,lower,upper,count,mean_confidence,accuracy
0,0.0,0.1,0,NaN,NaN
1,0.1,0.2,0,NaN,NaN
2,0.2,0.3,1,0.237117,0.000000
3,0.3,0.4,2,0.364303,0.000000
4,0.4,0.5,11,0.455224,0.181818
5,0.5,0.6,42,0.542860,0.333333
6,0.6,0.7,40,0.643344,0.400000
7,0.7,0.8,46,0.755804,0.456522
8,0.8,0.9,52,0.853032,0.480769
9,0.9,1.0,376,0.982888,0.702128


## Deliverable 1 — Test Set Evaluation Report

**Overall top-1 accuracy:** 60.00% (342/570 test frames).

The frozen checkpoint correctly classifies 60.00% of frames from 4 held-out public participants using the original side-view split. This measured frame-level result does not establish reliability on Aaron's online clients, other camera views, or form-assessment decisions.

**Per-class accuracy (correct / true-class support):**

| class | test_frames | correct | accuracy |
| --- | --- | --- | --- |
| dumbbell_side_bend | 96 | 83 | 86.46% |
| bent_over_row | 72 | 66 | 91.67% |
| bicep_curl | 72 | 15 | 20.83% |
| push_up | 60 | 36 | 60.00% |
| shoulder_press | 90 | 74 | 82.22% |
| squat | 84 | 25 | 29.76% |
| triceps_extension | 96 | 43 | 44.79% |

The lowest observed class accuracy is 20.83% for bicep_curl, and the highest is 91.67% for bent_over_row. Deployment review must account for this uneven class reliability and the displayed support rather than applying the aggregate score to every exercise.

**Confusion matrix:** see the rendered confusion_matrix.png figure above.

The largest off-diagonal confusion is bicep_curl to dumbbell_side_bend, with 49 incorrect test frames. These confusions can route a client's frame to the wrong exercise category, so downstream components need to preserve uncertainty and permit trainer correction.

**Reliability diagram:** see the rendered reliability_diagram.png figure above.

The top-label reliability bins yield ECE 28.28% and a signed mean-confidence minus accuracy gap of +28.28 percentage points. These descriptive calibration results do not make an individual softmax score a guarantee of correctness, and sparse or empty bins limit conclusions about their confidence ranges.

In [10]:
observed = per_class.loc[per_class.test_frames.gt(0)]
weak = observed.loc[observed.accuracy.eq(observed.accuracy.min()), "class"].tolist()
strong = observed.loc[observed.accuracy.eq(observed.accuracy.max()), "class"].tolist()
error_pairs = Counter((CLASSES[true_indices[i]], CLASSES[predicted[i]])
                      for i in np.flatnonzero(~correct))
if error_pairs:
    (largest_true, largest_pred), largest_count = error_pairs.most_common(1)[0]
    confusion_sentence = (f"The largest off-diagonal confusion is {largest_true} to "
                          f"{largest_pred}, with {largest_count} incorrect test frames.")
else:
    confusion_sentence = "The matrix has no observed off-diagonal errors in this finite test set."
evaluation_md = (
    "## Deliverable 1 — Test Set Evaluation Report\n\n"
    f"**Overall top-1 accuracy:** {metrics['top1_accuracy']:.2%} "
    f"({int(correct.sum())}/{len(test)} test frames).\n\n"
    f"The frozen checkpoint correctly classifies {metrics['top1_accuracy']:.2%} of frames "
    f"from {test.client_id.nunique()} held-out public participants using the original side-view split. "
    "This measured frame-level result does not establish reliability on Aaron's online clients, "
    "other camera views, or form-assessment decisions.\n\n"
    "**Per-class accuracy (correct / true-class support):**\n\n"
    + markdown_table(per_class, ["accuracy"]) + "\n\n"
    f"The lowest observed class accuracy is {observed.accuracy.min():.2%} for {', '.join(weak)}, "
    f"and the highest is {observed.accuracy.max():.2%} for {', '.join(strong)}. "
    "Deployment review must account for this uneven class reliability and the displayed support "
    "rather than applying the aggregate score to every exercise.\n\n"
    "**Confusion matrix:** see the rendered confusion_matrix.png figure above.\n\n"
    + confusion_sentence + " These confusions can route a client's frame to the wrong exercise category, "
    "so downstream components need to preserve uncertainty and permit trainer correction.\n\n"
    "**Reliability diagram:** see the rendered reliability_diagram.png figure above.\n\n"
    f"The top-label reliability bins yield ECE {metrics['ece']:.2%} and a signed mean-confidence "
    f"minus accuracy gap of {100 * metrics['signed_confidence_gap']:+.2f} percentage points. "
    "These descriptive calibration results do not make an individual softmax score a guarantee of "
    "correctness, and sparse or empty bins limit conclusions about their confidence ranges."
)
if per_class.test_frames.eq(0).any():
    evaluation_md += "\n\n**Coverage limitation:** test support is zero for " + ", ".join(
        per_class.loc[per_class.test_frames.eq(0), "class"]) + "; those class accuracies are not evaluated."
validation_accuracy = float(m4["stages"][1]["selected_validation_accuracy"])
display(Markdown(evaluation_md))
print(f"M4 selected validation accuracy: {validation_accuracy:.2%}; "
      f"test minus validation: {100 * (metrics['top1_accuracy'] - validation_accuracy):+.2f} pp.")

## Deliverable 1 — Test Set Evaluation Report

**Overall top-1 accuracy:** 60.00% (342/570 test frames).

The frozen checkpoint correctly classifies 60.00% of frames from 4 held-out public participants using the original side-view split. This measured frame-level result does not establish reliability on Aaron's online clients, other camera views, or form-assessment decisions.

**Per-class accuracy (correct / true-class support):**

| class | test_frames | correct | accuracy |
| --- | --- | --- | --- |
| dumbbell_side_bend | 96 | 83 | 86.46% |
| bent_over_row | 72 | 66 | 91.67% |
| bicep_curl | 72 | 15 | 20.83% |
| push_up | 60 | 36 | 60.00% |
| shoulder_press | 90 | 74 | 82.22% |
| squat | 84 | 25 | 29.76% |
| triceps_extension | 96 | 43 | 44.79% |

The lowest observed class accuracy is 20.83% for bicep_curl, and the highest is 91.67% for bent_over_row. Deployment review must account for this uneven class reliability and the displayed support rather than applying the aggregate score to every exercise.

**Confusion matrix:** see the rendered confusion_matrix.png figure above.

The largest off-diagonal confusion is bicep_curl to dumbbell_side_bend, with 49 incorrect test frames. These confusions can route a client's frame to the wrong exercise category, so downstream components need to preserve uncertainty and permit trainer correction.

**Reliability diagram:** see the rendered reliability_diagram.png figure above.

The top-label reliability bins yield ECE 28.28% and a signed mean-confidence minus accuracy gap of +28.28 percentage points. These descriptive calibration results do not make an individual softmax score a guarantee of correctness, and sparse or empty bins limit conclusions about their confidence ranges.

M4 selected validation accuracy: 71.56%; test minus validation: -11.56 pp.


## Part 2 — Systematic review of 30 real errors
A fixed seed selects 30 distinct incorrect test frames without replacement. If fewer errors exist, inspect all available errors and report the actual denominator; do not worsen the model, change the test distribution, or invent errors to reach 30. Frequencies in the error report refer to this reviewed sample, not to the entire test set.

The three most common true-to-predicted confusions in the sample are quantitative candidates for three error patterns. Use the images to write a falsifiable cause hypothesis and one concrete data, architecture, or training recommendation for each. If visual patterns are more useful, specify three custom patterns using actual sampled frame IDs in the observation cell. Patterns may overlap and need not cover every sampled error.

In [11]:
sample_indices = select_error_indices(correct, seed=SEED + 600, requested=ERROR_SAMPLE_SIZE)
error_sample = predictions.iloc[sample_indices].reset_index(drop=True)
error_sample.to_csv(OUTPUT / "reviewed_errors.csv", index=False)
sample_pairs = Counter(zip(error_sample.label, error_sample.predicted_class))
proposed_patterns = []
for (true_label, predicted_label), count in sample_pairs.most_common(3):
    frame_ids = error_sample.loc[
        error_sample.label.eq(true_label) & error_sample.predicted_class.eq(predicted_label),
        "frame_id"].tolist()
    proposed_patterns.append({
        "pattern": f"{true_label} frames are incorrectly classified as {predicted_label}",
        "frame_ids": frame_ids,
    })
print(f"Total test errors: {metrics['test_errors']}/{len(test)}. "
      f"Review sample: {len(error_sample)}/{ERROR_SAMPLE_SIZE} requested errors.")
if len(error_sample) < ERROR_SAMPLE_SIZE:
    print("Sample-size shortfall: disclose the actual available error count in the submission.")
if len(proposed_patterns) < 3:
    print("Fewer than three confusion types. Do not invent a third; inspect actual visual patterns.")
display(error_sample[["frame_id", "label", "predicted_class", "confidence", "client_id", "video_path"]])
print("Candidate pattern definitions:")
pprint.pprint(proposed_patterns, sort_dicts=False)
for start in range(0, len(error_sample), 6):
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, error_sample.iloc[start:start + 6].itertuples(index=False)):
        ax.imshow(load_rgb(resolve_path(row.path)))
        ax.set_title(f"{row.frame_id}\nTrue: {row.label}\n"
                     f"Pred: {row.predicted_class} ({row.confidence:.3f})", fontsize=8)
    fig.suptitle(f"Reviewed errors {start + 1}–{min(start + 6, len(error_sample))}")
    fig.tight_layout()
    fig.savefig(OUTPUT / f"errors_{start + 1:02d}.png", dpi=150)
    plt.show()

Total test errors: 228/570. Review sample: 30/30 requested errors.


,frame_id,label,predicted_class,confidence,client_id,video_path
0,3d51ad022e8f430f_00000317,triceps_extension,shoulder_press,0.754325,public_subject_008,videos/public_subject_008/subject_008_tricep_b...
1,ee179cc21124f650_00000378,triceps_extension,shoulder_press,0.806126,public_subject_026,videos/public_subject_026/subject_026_tricep_g...
2,1ae5cf90793603cd_00000271,dumbbell_side_bend,squat,0.940490,public_subject_010,videos/public_subject_010/subject_010_abs_bad_...
3,0d929360090f5e3a_00000113,push_up,bent_over_row,0.982631,public_subject_018,videos/public_subject_018/subject_018_push_up_...
4,da33ccaa00e632ff_00000347,bicep_curl,dumbbell_side_bend,0.638433,public_subject_010,videos/public_subject_010/subject_010_bicep_cu...
5,21014302d231ec00_00000196,squat,bicep_curl,0.747558,public_subject_018,videos/public_subject_018/subject_018_squat_go...
6,ee179cc21124f650_00000273,triceps_extension,shoulder_press,0.964141,public_subject_026,videos/public_subject_026/subject_026_tricep_g...
7,db6e552780992f9f_00000049,bicep_curl,bent_over_row,0.860873,public_subject_010,videos/public_subject_010/subject_010_bicep_cu...
8,0d929360090f5e3a_00000128,push_up,bent_over_row,0.787940,public_subject_018,videos/public_subject_018/subject_018_push_up_...
9,0822fd5dd9560449_00000241,shoulder_press,triceps_extension,0.963903,public_subject_018,videos/public_subject_018/subject_018_shoulder...


Candidate pattern definitions:
[{'pattern': 'triceps_extension frames are incorrectly classified as '
             'shoulder_press',
  'frame_ids': ['3d51ad022e8f430f_00000317',
                'ee179cc21124f650_00000378',
                'ee179cc21124f650_00000273',
                'ee179cc21124f650_00000183',
                'ee179cc21124f650_00000048']},
 {'pattern': 'bicep_curl frames are incorrectly classified as '
             'dumbbell_side_bend',
  'frame_ids': ['da33ccaa00e632ff_00000347',
                '7b2cb21d20afbe9f_00000364',
                'c20df6295c262aad_00000346',
                'c9b2a305e0a02ad2_00000080']},
 {'pattern': 'squat frames are incorrectly classified as bent_over_row',
  'frame_ids': ['5b3860e6780d7de2_00000074',
                'bac4ac5a2275de66_00000330',
                '5b3860e6780d7de2_00000044']}]


Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

In [12]:
def gradcam_for_class(maps, class_index):
    # Watch the final backbone tensor directly. The head is the verified original GAP+Dense
    # path, with Dropout disabled; logits avoid saturated softmax gradients.
    maps = tf.convert_to_tensor(maps, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(maps)
        target = logits_from_maps(maps)[:, class_index]
    gradients = tape.gradient(target, maps)
    if gradients is None or not np.isfinite(gradients.numpy()).all():
        raise RuntimeError("GradCAM gradient is disconnected or non-finite.")
    channel_weights = tf.reduce_mean(gradients, axis=(0, 1, 2))
    heatmap = tf.nn.relu(tf.reduce_sum(maps[0] * channel_weights, axis=-1))
    maximum = float(tf.reduce_max(heatmap))
    return (heatmap.numpy() / maximum if maximum > 0 else np.zeros((7, 7), np.float32)), maximum

gradcam_rows = error_sample.sort_values("confidence", ascending=False).head(GRADCAM_COUNT)
gradcam_ids = gradcam_rows.frame_id.tolist()
gradcam_records = []
for row in gradcam_rows.itertuples(index=False):
    rgb = load_rgb(resolve_path(row.path))
    model_image = tf.image.resize(tf.cast(rgb, tf.float32), INPUT_SIZE, method="bilinear").numpy() / 255
    maps = feature_model(preprocess_rgb(rgb), training=False)
    predicted_cam, predicted_peak = gradcam_for_class(maps, label_to_index[row.predicted_class])
    true_cam, true_peak = gradcam_for_class(maps, label_to_index[row.label])
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(np.clip(model_image, 0, 1))
    axes[0].set_title("Resized input")
    for ax, heatmap, title in [
        (axes[1], predicted_cam, f"Predicted: {row.predicted_class}"),
        (axes[2], true_cam, f"True: {row.label}"),
    ]:
        ax.imshow(np.clip(model_image, 0, 1))
        ax.imshow(tf.image.resize(heatmap[..., None], INPUT_SIZE).numpy()[..., 0],
                  cmap="magma", alpha=0.45, vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{row.frame_id} · incorrect confidence {row.confidence:.3f}")
    fig.tight_layout()
    figure_name = f"gradcam_{row.frame_id}.png"
    fig.savefig(OUTPUT / figure_name, dpi=150)
    plt.show()
    gradcam_records.append({
        "frame_id": row.frame_id, "predicted_class": row.predicted_class,
        "true_class": row.label, "confidence": float(row.confidence), "figure": figure_name,
        "predicted_cam_nonzero": predicted_peak > 0, "true_cam_nonzero": true_peak > 0,
    })
pd.DataFrame(gradcam_records).to_csv(OUTPUT / "gradcam_cases.csv", index=False)
print("Review these IDs:", gradcam_ids)
print("GradCAM is attribution evidence, not causal proof or a segmentation mask. "
      "An all-zero map is inconclusive, not evidence of valid-region reasoning.")

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Figure retained in the original executed notebook export.

Review these IDs: ['c9b2a305e0a02ad2_00000080', '7b2cb21d20afbe9f_00000364', 'c20df6295c262aad_00000346', 'bac4ac5a2275de66_00000195', '16850e6324b9dd1f_00000137', 'bac4ac5a2275de66_00000300']
GradCAM is attribution evidence, not causal proof or a segmentation mask. An all-zero map is inconclusive, not evidence of valid-region reasoning.


### Review of the sampled errors and GradCAM overlays

I grouped the reviewed images by three recurring confusions and tagged the six predicted-class overlays by the regions they highlighted. The hypotheses and recommendations in Deliverable 2 refer to the displayed frame IDs.


In [13]:
# Visual review of the saved error frames and predicted-class GradCAM overlays.
ERROR_PATTERN_NOTES = {1: {'hypothesized_cause': 'The raised upper arm and dumbbell near or behind the head in frames '
                           '3d51ad022e8f430f_00000317 and ee179cc21124f650_00000273 give both '
                           'exercises a similar side-view outline. A single frame can miss how '
                           'much of the movement comes from the elbow versus the shoulder',
     'recommendation': 'Train a short-clip classifier on paired examples of both exercises across '
                       'a full repetition so it can learn differences in elbow and upper-arm '
                       'motion'},
 2: {'hypothesized_cause': 'In frames 7b2cb21d20afbe9f_00000364 and c9b2a305e0a02ad2_00000080, the '
                           'lifter stands with a dumbbell beside the thigh. The model may read '
                           'that position as a side bend when a single image shows little elbow '
                           'flexion',
     'recommendation': 'Annotate active, bottom-of-rep, and idle frames for curls and side bends. '
                       'Curate unambiguous active poses for exercise supervision and retain '
                       'ambiguous poses as examples to defer for review'},
 3: {'hypothesized_cause': 'The forward trunk lean in frames 5b3860e6780d7de2_00000044 and '
                           '5b3860e6780d7de2_00000074 resembles a bent-over row from the side, '
                           'even though both frames show flexed knees and lowered hips. The '
                           'classifier may favor trunk angle over those lower-body cues',
     'recommendation': 'Add matched squat and row examples with similar trunk angles to a future '
                       'training split, using full-body crops that preserve the knees and hips'}}
CUSTOM_ERROR_PATTERNS = [{'pattern': 'Frames labeled as triceps extension are predicted as shoulder press',
  'frame_ids': ['3d51ad022e8f430f_00000317',
                'ee179cc21124f650_00000378',
                'ee179cc21124f650_00000273',
                'ee179cc21124f650_00000183',
                'ee179cc21124f650_00000048']},
 {'pattern': 'Frames labeled as bicep curl are predicted as dumbbell side bend',
  'frame_ids': ['da33ccaa00e632ff_00000347',
                '7b2cb21d20afbe9f_00000364',
                'c20df6295c262aad_00000346',
                'c9b2a305e0a02ad2_00000080']},
 {'pattern': 'Frames labeled as squat are predicted as bent-over row',
  'frame_ids': ['5b3860e6780d7de2_00000074',
                'bac4ac5a2275de66_00000330',
                '5b3860e6780d7de2_00000044']}]
GRADCAM_REVIEWS = {'c9b2a305e0a02ad2_00000080': 'valid_region',
 '7b2cb21d20afbe9f_00000364': 'valid_region',
 'c20df6295c262aad_00000346': 'spurious_attention',
 'bac4ac5a2275de66_00000195': 'mixed',
 '16850e6324b9dd1f_00000137': 'mixed',
 'bac4ac5a2275de66_00000300': 'spurious_attention'}
GRADCAM_CONCLUSION = ('The predicted-class overlays show a mix. In c9b2a305e0a02ad2_00000080 and '
 '7b2cb21d20afbe9f_00000364, the response covers the working arm and dumbbell, yet the classifier '
 'calls both curls side bends. In c20df6295c262aad_00000346 and bac4ac5a2275de66_00000300, the '
 'strongest patch sits on the bench and adjacent floor to the right of a blurred lifter. The maps '
 'for bac4ac5a2275de66_00000195 and 16850e6324b9dd1f_00000137 span the lower legs and nearby '
 'equipment. These six high-confidence errors show that the model responds to plausible exercise '
 'cues and can also highlight the surrounding scene. The coarse overlays support a '
 'background-shortcut hypothesis, which would need a controlled masking test to establish '
 'causality.')
DOWNSTREAM_PATTERN_NUMBERS = [2, 3]
print("Recorded three error patterns and reviewed six GradCAM cases.")

Recorded three error patterns and reviewed six GradCAM cases.


## Deliverable 2 — Error Analysis Summary

228 total test errors; a seeded sample of 30 distinct errors was selected for visual review.

**Pattern 1:** Frames labeled as triceps extension are predicted as shoulder press in 5/30 reviewed errors. Hypothesized cause: The raised upper arm and dumbbell near or behind the head in frames 3d51ad022e8f430f_00000317 and ee179cc21124f650_00000273 give both exercises a similar side-view outline. A single frame can miss how much of the movement comes from the elbow versus the shoulder. Concrete recommendation: Train a short-clip classifier on paired examples of both exercises across a full repetition so it can learn differences in elbow and upper-arm motion.

Evidence frame IDs: 3d51ad022e8f430f_00000317, ee179cc21124f650_00000378, ee179cc21124f650_00000273, ee179cc21124f650_00000183, ee179cc21124f650_00000048.

**Pattern 2:** Frames labeled as bicep curl are predicted as dumbbell side bend in 4/30 reviewed errors. Hypothesized cause: In frames 7b2cb21d20afbe9f_00000364 and c9b2a305e0a02ad2_00000080, the lifter stands with a dumbbell beside the thigh. The model may read that position as a side bend when a single image shows little elbow flexion. Concrete recommendation: Annotate active, bottom-of-rep, and idle frames for curls and side bends. Curate unambiguous active poses for exercise supervision and retain ambiguous poses as examples to defer for review.

Evidence frame IDs: da33ccaa00e632ff_00000347, 7b2cb21d20afbe9f_00000364, c20df6295c262aad_00000346, c9b2a305e0a02ad2_00000080.

**Pattern 3:** Frames labeled as squat are predicted as bent-over row in 3/30 reviewed errors. Hypothesized cause: The forward trunk lean in frames 5b3860e6780d7de2_00000044 and 5b3860e6780d7de2_00000074 resembles a bent-over row from the side, even though both frames show flexed knees and lowered hips. The classifier may favor trunk angle over those lower-body cues. Concrete recommendation: Add matched squat and row examples with similar trunk angles to a future training split, using full-body crops that preserve the knees and hips.

Evidence frame IDs: 5b3860e6780d7de2_00000074, bac4ac5a2275de66_00000330, 5b3860e6780d7de2_00000044.

**GradCAM interpretation conclusion:**

The predicted-class overlays show a mix. In c9b2a305e0a02ad2_00000080 and 7b2cb21d20afbe9f_00000364, the response covers the working arm and dumbbell, yet the classifier calls both curls side bends. In c20df6295c262aad_00000346 and bac4ac5a2275de66_00000300, the strongest patch sits on the bench and adjacent floor to the right of a blurred lifter. The maps for bac4ac5a2275de66_00000195 and 16850e6324b9dd1f_00000137 span the lower legs and nearby equipment. These six high-confidence errors show that the model responds to plausible exercise cues and can also highlight the surrounding scene. The coarse overlays support a background-shortcut hypothesis, which would need a controlled masking test to establish causality. The reviewed attribution tags are {'valid_region': 2, 'spurious_attention': 2, 'mixed': 2} across 6 cases; these heatmaps support hypotheses but do not prove a causal feature dependency.

In [14]:
chosen_patterns = CUSTOM_ERROR_PATTERNS or proposed_patterns
reviewed_ids = set(error_sample.frame_id)
error_details = []
for number, pattern in enumerate(chosen_patterns, 1):
    frame_ids = pattern.get("frame_ids", [])
    if (not frame_ids or len(set(frame_ids)) != len(frame_ids)
        or not set(frame_ids).issubset(reviewed_ids)):
        raise ValueError("Each pattern must name distinct actual IDs from the reviewed error sample.")
    note = ERROR_PATTERN_NOTES.get(number, {})
    error_details.append({
        "number": number, "pattern": pattern.get("pattern", "").strip(),
        "count": len(frame_ids), "denominator": len(error_sample), "frame_ids": frame_ids,
        "hypothesized_cause": note.get("hypothesized_cause", "").strip(),
        "recommendation": note.get("recommendation", "").strip(),
    })
patterns_complete = len(error_details) == 3 and all(
    d["pattern"] and d["hypothesized_cause"] and d["recommendation"] for d in error_details)
allowed_tags = {"valid_region", "spurious_attention", "mixed", "inconclusive"}
if any(tag and tag not in allowed_tags for tag in GRADCAM_REVIEWS.values()):
    raise ValueError(f"Use GradCAM tags from {sorted(allowed_tags)}.")
if set(GRADCAM_REVIEWS) != set(gradcam_ids):
    raise ValueError("GradCAM review IDs must match the displayed cases.")
tagged = {i: GRADCAM_REVIEWS[i] for i in gradcam_ids if GRADCAM_REVIEWS[i] in allowed_tags}
cam_complete = bool(gradcam_ids) and len(tagged) == len(gradcam_ids) and bool(GRADCAM_CONCLUSION.strip())
error_md = "## Deliverable 2 — Error Analysis Summary\n\n"
error_md += (f"{metrics['test_errors']} total test errors; a seeded sample of "
             f"{len(error_sample)} distinct errors was selected for visual review.\n\n")
if len(error_sample) != ERROR_SAMPLE_SIZE:
    error_md += (f"**Sample limitation:** only {len(error_sample)} actual errors were available "
                 f"for the requested {ERROR_SAMPLE_SIZE}-error review. No errors were manufactured.\n\n")
for detail in error_details:
    error_md += (
        f"**Pattern {detail['number']}:** {detail['pattern']} in "
        f"{detail['count']}/{detail['denominator']} reviewed errors. "
        f"Hypothesized cause: {detail['hypothesized_cause'] or 'AWAITING VISUAL REVIEW'}. "
        f"Concrete recommendation: {detail['recommendation'] or 'AWAITING VISUAL REVIEW'}.\n\n"
        f"Evidence frame IDs: {', '.join(detail['frame_ids'])}.\n\n")
error_md += "**GradCAM interpretation conclusion:**\n\n"
if cam_complete:
    tag_counts = Counter(tagged.values())
    error_md += (GRADCAM_CONCLUSION.strip() +
                 f" The reviewed attribution tags are {dict(tag_counts)} across {len(tagged)} cases; "
                 "these heatmaps support hypotheses but do not prove a causal feature dependency.")
else:
    error_md += "AWAITING VISUAL REVIEW: complete all displayed case tags and the conclusion paragraph."
if not patterns_complete:
    error_md += "\n\n**Incomplete:** exactly three evidence-backed patterns with causes and recommendations are required."
if (len(DOWNSTREAM_PATTERN_NUMBERS) != 2 or len(set(DOWNSTREAM_PATTERN_NUMBERS)) != 2
    or any(n not in [1, 2, 3] for n in DOWNSTREAM_PATTERN_NUMBERS)):
    raise ValueError("Select two distinct downstream pattern numbers from 1, 2, 3.")
downstream_limitations = []
for number in DOWNSTREAM_PATTERN_NUMBERS:
    detail = next((d for d in error_details if d["number"] == number), None)
    downstream_limitations.append(
        f"Pattern {number}: {detail['pattern']} ({detail['count']}/{detail['denominator']} reviewed errors). "
        f"Hypothesis: {detail['hypothesized_cause'] or 'pending review'}. "
        f"Recommended change: {detail['recommendation'] or 'pending review'}."
        if detail else f"Pattern {number}: pending actual evidence.")
display(Markdown(error_md))

## Deliverable 2 — Error Analysis Summary

228 total test errors; a seeded sample of 30 distinct errors was selected for visual review.

**Pattern 1:** Frames labeled as triceps extension are predicted as shoulder press in 5/30 reviewed errors. Hypothesized cause: The raised upper arm and dumbbell near or behind the head in frames 3d51ad022e8f430f_00000317 and ee179cc21124f650_00000273 give both exercises a similar side-view outline. A single frame can miss how much of the movement comes from the elbow versus the shoulder. Concrete recommendation: Train a short-clip classifier on paired examples of both exercises across a full repetition so it can learn differences in elbow and upper-arm motion.

Evidence frame IDs: 3d51ad022e8f430f_00000317, ee179cc21124f650_00000378, ee179cc21124f650_00000273, ee179cc21124f650_00000183, ee179cc21124f650_00000048.

**Pattern 2:** Frames labeled as bicep curl are predicted as dumbbell side bend in 4/30 reviewed errors. Hypothesized cause: In frames 7b2cb21d20afbe9f_00000364 and c9b2a305e0a02ad2_00000080, the lifter stands with a dumbbell beside the thigh. The model may read that position as a side bend when a single image shows little elbow flexion. Concrete recommendation: Annotate active, bottom-of-rep, and idle frames for curls and side bends. Curate unambiguous active poses for exercise supervision and retain ambiguous poses as examples to defer for review.

Evidence frame IDs: da33ccaa00e632ff_00000347, 7b2cb21d20afbe9f_00000364, c20df6295c262aad_00000346, c9b2a305e0a02ad2_00000080.

**Pattern 3:** Frames labeled as squat are predicted as bent-over row in 3/30 reviewed errors. Hypothesized cause: The forward trunk lean in frames 5b3860e6780d7de2_00000044 and 5b3860e6780d7de2_00000074 resembles a bent-over row from the side, even though both frames show flexed knees and lowered hips. The classifier may favor trunk angle over those lower-body cues. Concrete recommendation: Add matched squat and row examples with similar trunk angles to a future training split, using full-body crops that preserve the knees and hips.

Evidence frame IDs: 5b3860e6780d7de2_00000074, bac4ac5a2275de66_00000330, 5b3860e6780d7de2_00000044.

**GradCAM interpretation conclusion:**

The predicted-class overlays show a mix. In c9b2a305e0a02ad2_00000080 and 7b2cb21d20afbe9f_00000364, the response covers the working arm and dumbbell, yet the classifier calls both curls side bends. In c20df6295c262aad_00000346 and bac4ac5a2275de66_00000300, the strongest patch sits on the bench and adjacent floor to the right of a blurred lifter. The maps for bac4ac5a2275de66_00000195 and 16850e6324b9dd1f_00000137 span the lower legs and nearby equipment. These six high-confidence errors show that the model responds to plausible exercise cues and can also highlight the surrounding scene. The coarse overlays support a background-shortcut hypothesis, which would need a controlled masking test to establish causality. The reviewed attribution tags are {'valid_region': 2, 'spurious_attention': 2, 'mixed': 2} across 6 cases; these heatmaps support hypotheses but do not prove a causal feature dependency.

## Part 3 — Representation space and nearest neighbors
Extract the exact GAP vectors used by the frozen classification head. For visualization, L2-normalize them, reduce to at most 50 dimensions with PCA, and fit seeded t-SNE. Color the same projection by exercise and participant ID to inspect possible scene/identity organization without inferring demographic traits. This exploratory fit does not update the vision model.

Neighbors are computed by cosine distance in the original 2,048-dimensional representation, not by distance in the t-SNE picture. Show unrestricted neighbors and neighbors from another source clip so adjacent frames cannot dominate the interpretation unnoticed. The five displayed comparisons use the cross-clip neighbors. Cross-clip neighbors may still show the same participant; report that metadata explicitly.

In [15]:
if not np.isfinite(embeddings).all() or np.any(np.linalg.norm(embeddings, axis=1) == 0):
    raise ValueError("Non-finite or zero embeddings prevent a meaningful representation audit.")
normalized_embeddings = normalize(embeddings, norm="l2")
pca_dimensions = min(50, len(test) - 1, embeddings.shape[1])
reduced = PCA(n_components=pca_dimensions, random_state=SEED).fit_transform(normalized_embeddings)
perplexity = min(30.0, float(max(1, (len(test) - 1) // 3)))
tsne_kwargs = {"n_components": 2, "perplexity": perplexity, "init": "pca",
               "learning_rate": "auto", "random_state": SEED}
iteration_arg = "max_iter" if "max_iter" in inspect.signature(TSNE).parameters else "n_iter"
tsne_kwargs[iteration_arg] = 1000
projection = TSNE(**tsne_kwargs).fit_transform(reduced)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for label in CLASSES:
    mask = test.label.eq(label).to_numpy()
    axes[0].scatter(projection[mask, 0], projection[mask, 1], s=20, alpha=0.65,
                    label=label.replace("_", " "))
for participant in sorted(test.client_id.unique()):
    mask = test.client_id.eq(participant).to_numpy()
    axes[1].scatter(projection[mask, 0], projection[mask, 1], s=20, alpha=0.65, label=participant)
axes[0].set_title("Same projection · true exercise")
axes[1].set_title("Same projection · source participant")
for ax in axes:
    ax.set(xlabel="t-SNE dimension 1", ylabel="t-SNE dimension 2")
    ax.legend(fontsize=8)
fig.suptitle(f"All {len(test)} test-frame GAP embeddings · PCA {pca_dimensions}D · perplexity {perplexity:g}")
fig.tight_layout()
fig.savefig(OUTPUT / "representation_tsne.png", dpi=160)
plt.show()
projection_table = test[["frame_id", "label", "client_id", "video_path"]].copy()
projection_table["tsne_1"], projection_table["tsne_2"] = projection[:, 0], projection[:, 1]
projection_table.to_csv(OUTPUT / "tsne_projection.csv", index=False)
print("t-SNE configuration:", tsne_kwargs)
print("Cluster distances and apparent separation in this 2D view are not calibrated similarity scores.")

Figure retained in the original executed notebook export.

t-SNE configuration: {'n_components': 2, 'perplexity': 30.0, 'init': 'pca', 'learning_rate': 'auto', 'random_state': 42, 'max_iter': 1000}
Cluster distances and apparent separation in this 2D view are not calibrated similarity scores.


In [16]:
query_indices = np.random.default_rng(SEED + 601).choice(
    len(test), size=min(NN_QUERY_COUNT, len(test)), replace=False)
nn_records = []
for index in query_indices:
    distances = cosine_distances(embeddings[index:index + 1], embeddings).ravel()
    distances[index] = np.inf
    nearest = int(distances.argmin())
    candidates = test.video_path.ne(test.iloc[index].video_path).to_numpy()
    cross_distances = np.where(candidates, distances, np.inf)
    cross_index = int(cross_distances.argmin()) if np.isfinite(cross_distances).any() else None
    nn_records.append({
        "query_frame_id": test.iloc[index].frame_id, "query_class": test.iloc[index].label,
        "nearest_any_frame_id": test.iloc[nearest].frame_id,
        "nearest_any_class": test.iloc[nearest].label,
        "nearest_any_same_clip": bool(test.iloc[nearest].video_path == test.iloc[index].video_path),
        "cross_clip_frame_id": test.iloc[cross_index].frame_id if cross_index is not None else "",
        "cross_clip_class": test.iloc[cross_index].label if cross_index is not None else "",
        "cross_clip_cosine_distance": float(distances[cross_index]) if cross_index is not None else np.nan,
        "cross_clip_same_participant": bool(test.iloc[cross_index].client_id == test.iloc[index].client_id)
                                     if cross_index is not None else None,
    })
neighbor_table = pd.DataFrame(nn_records)
neighbor_table.to_csv(OUTPUT / "nearest_neighbors.csv", index=False)
display(neighbor_table)
fig, axes = plt.subplots(2, len(nn_records), figsize=(3.3 * len(nn_records), 6), squeeze=False)
by_id = test.set_index("frame_id")
for column, pair in enumerate(nn_records):
    for row_number, frame_key, label_key in [
        (0, "query_frame_id", "query_class"), (1, "cross_clip_frame_id", "cross_clip_class")]:
        ax = axes[row_number, column]
        frame_id = pair[frame_key]
        if frame_id:
            ax.imshow(load_rgb(by_id.loc[frame_id, "resolved_path"]))
            ax.set_title(f"{'Query' if row_number == 0 else 'Other-clip neighbor'}\n"
                         f"{pair[label_key]}\n{frame_id}", fontsize=8)
        else:
            ax.text(0.5, 0.5, "No other-clip neighbor", ha="center", va="center")
        ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT / "nearest_neighbors.png", dpi=160)
plt.show()

,query_frame_id,query_class,nearest_any_frame_id,nearest_any_class,nearest_any_same_clip,cross_clip_frame_id,cross_clip_class,cross_clip_cosine_distance,cross_clip_same_participant
0,0d929360090f5e3a_00000023,push_up,0d929360090f5e3a_00000053,push_up,True,c19242117db5201e_00000144,push_up,0.142994,True
1,d4957212a41337f8_00000271,triceps_extension,d4957212a41337f8_00000241,triceps_extension,True,fb533c76c8433dbb_00000071,shoulder_press,0.064337,True
2,edfaac4a19a7fbac_00000300,dumbbell_side_bend,edfaac4a19a7fbac_00000240,dumbbell_side_bend,True,f4a6c60b6f9e0bec_00000046,dumbbell_side_bend,0.072064,True
3,b997443c65e3b27f_00000312,dumbbell_side_bend,b997443c65e3b27f_00000162,dumbbell_side_bend,True,105f97a407375bab_00000104,dumbbell_side_bend,0.107391,True
4,16096756cb339daa_00000253,shoulder_press,16096756cb339daa_00000343,shoulder_press,True,0822fd5dd9560449_00000346,shoulder_press,0.066794,True


Figure retained in the original executed notebook export.

### Review of the projection and nearest neighbors

I compared the exercise and participant colorings of the same t-SNE projection and checked the five displayed cross-clip neighbor pairs. Deliverable 3 uses these observations to assess how the representation might transfer to different clients.


In [17]:
REPRESENTATION_SENTENCES = ['The t-SNE plot shows several compact islands for push-ups, and the same projection colored by '
 'participant has large regions dominated by one person.',
 'All five cross-clip neighbors came from the same participant as the query, and four shared its '
 'true exercise label, suggesting that the embeddings reflect both posture and participant or '
 'recording cues.',
 'Posture similarity is useful for trainer review, while concentration by participant could hurt '
 'transfer to clients recording in different settings.',
 'The triceps extension query `d4957212a41337f8_00000271` retrieves shoulder press frame '
 '`fb533c76c8433dbb_00000071` from another clip of the same participant, with a raised upper arm '
 'and a dumbbell near the head in both frames.',
 'The two-dimensional projection and the five neighbor queries support testing the effect of '
 'appearance and background in a separate cross-participant comparison.',
 'I consider the current structure a limitation for cross-client use because two different '
 'exercises can land close together when their poses and recording setup look similar.']
print("Recorded six sentences interpreting the projection and nearest neighbors.")

Recorded six sentences interpreting the projection and nearest neighbors.


## Deliverable 3 — Representation Space Interpretation

The t-SNE plot shows several compact islands for push-ups, and the same projection colored by participant has large regions dominated by one person. All five cross-clip neighbors came from the same participant as the query, and four shared its true exercise label, suggesting that the embeddings reflect both posture and participant or recording cues. Posture similarity is useful for trainer review, while concentration by participant could hurt transfer to clients recording in different settings. The triceps extension query `d4957212a41337f8_00000271` retrieves shoulder press frame `fb533c76c8433dbb_00000071` from another clip of the same participant, with a raised upper arm and a dumbbell near the head in both frames. The two-dimensional projection and the five neighbor queries support testing the effect of appearance and background in a separate cross-participant comparison. I consider the current structure a limitation for cross-client use because two different exercises can land close together when their poses and recording setup look similar.

In [18]:
representation_complete = (
    len(REPRESENTATION_SENTENCES) == 6
    and all(sentence.strip() for sentence in REPRESENTATION_SENTENCES)
    and 5 <= sentence_count(" ".join(REPRESENTATION_SENTENCES)) <= 7)
representation_md = "## Deliverable 3 — Representation Space Interpretation\n\n"
representation_md += (
    " ".join(sentence.strip() for sentence in REPRESENTATION_SENTENCES)
    if representation_complete else
    "AWAITING VISUAL REVIEW: fill all six sentence slots; the final paragraph must contain 5–7 sentences.")
display(Markdown(representation_md))

## Deliverable 3 — Representation Space Interpretation

The t-SNE plot shows several compact islands for push-ups, and the same projection colored by participant has large regions dominated by one person. All five cross-clip neighbors came from the same participant as the query, and four shared its true exercise label, suggesting that the embeddings reflect both posture and participant or recording cues. Posture similarity is useful for trainer review, while concentration by participant could hurt transfer to clients recording in different settings. The triceps extension query `d4957212a41337f8_00000271` retrieves shoulder press frame `fb533c76c8433dbb_00000071` from another clip of the same participant, with a raised upper arm and a dumbbell near the head in both frames. The two-dimensional projection and the five neighbor queries support testing the effect of appearance and background in a separate cross-participant comparison. I consider the current structure a limitation for cross-client use because two different exercises can land close together when their poses and recording setup look similar.

## Part 4 — Full model card
All nine rubric sections are included. Computed values come only from the verified checkpoint, manifests, and this evaluation. Unknown evidence is labeled explicitly and prevents the final completeness check from reporting success. Dataset quality labels are metadata; this classifier does not predict good/bad form.

In [19]:
source_form_table = predictions.groupby("source_form_label", sort=True).agg(
    test_frames=("frame_id", "size"), participants=("client_id", "nunique"),
    exercise_classification_accuracy=("correct", "mean")).reset_index()
participant_table = predictions.groupby("client_id", sort=True).agg(
    test_frames=("frame_id", "size"), source_clips=("video_path", "nunique"),
    exercise_classification_accuracy=("correct", "mean")).reset_index()
display(source_form_table)
display(participant_table)
source_form_table.to_csv(OUTPUT / "source_form_subgroups.csv", index=False)
participant_table.to_csv(OUTPUT / "participant_subgroups.csv", index=False)
print("These are exercise-recognition subgroup scores, not form-quality assessment scores.")
print("No age, gender, skin-tone, disability, or other demographic performance analysis was conducted.")

,source_form_label,test_frames,participants,exercise_classification_accuracy
0,bad,282,4,0.574468
1,good,288,4,0.625000


,client_id,test_frames,source_clips,exercise_classification_accuracy
0,public_subject_008,168,14,0.660714
1,public_subject_010,150,13,0.573333
2,public_subject_018,168,14,0.517857
3,public_subject_026,84,7,0.690476


These are exercise-recognition subgroup scores, not form-quality assessment scores.
No age, gender, skin-tone, disability, or other demographic performance analysis was conducted.


## Deliverable 4 — MIPDS Vision Module Model Card

### 1. Model Description
**Model/version:** MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d. **Task:** seven-class exercise recognition from side-view still frames.
**Architecture:** ImageNet-pretrained ResNet-50, final conv5_block3_out spatial map, global average pooling, inference-disabled Dropout, seven-way Dense softmax.
**Parameters:** 23,602,055 total; zero trainable parameters during this audit.
**Training completion date:** 2026-10-11 (checkpoint save date from the final M4 fine-tuning cell). **Checkpoint save timestamp:** 2026-10-11@23:41:45.
**Fine-tuning source:** Mendeley Data DOI 10.17632/kgbb3yn47p.3, Version 3, Subject-wise archive.

### 2. Intended Use
The module may supply exercise-category suggestions and visual features for human-reviewed trainer documentation within the evaluated side-view exercise domain.
It must not autonomously prescribe training, approve exercise safety, diagnose injury, label good/bad form, infer joint angles, count repetitions, or identify a client.
The output is an exercise category, not evidence that the movement was performed correctly; high confidence does not authorize an excluded use.

### 3. Performance — Aggregate
**Held-out top-1 frame accuracy:** 60.00% (342/570).
**Balanced accuracy:** 59.39%; **macro-F1:** 0.5765.
**Calibration:** top-label ECE 28.28% using 10 equal-width bins; signed mean-confidence gap +28.28 percentage points.
Confidence is raw softmax, not recalibrated on this test set. **mAP@0.5:** not evaluated; this classification adapter has no detection head or held-out box ground truth.

### 4. Performance — Disaggregated

| class | test_frames | correct | accuracy |
| --- | --- | --- | --- |
| dumbbell_side_bend | 96 | 83 | 86.46% |
| bent_over_row | 72 | 66 | 91.67% |
| bicep_curl | 72 | 15 | 20.83% |
| push_up | 60 | 36 | 60.00% |
| shoulder_press | 90 | 74 | 82.22% |
| squat | 84 | 25 | 29.76% |
| triceps_extension | 96 | 43 | 44.79% |

Lowest observed class accuracy: bicep_curl (20.83%).
Highest observed class accuracy: bent_over_row (91.67%).
Source-form-label and participant analyses were conducted for exercise recognition:

| source_form_label | test_frames | participants | exercise_classification_accuracy |
| --- | --- | --- | --- |
| bad | 282 | 4 | 57.45% |
| good | 288 | 4 | 62.50% |

| client_id | test_frames | source_clips | exercise_classification_accuracy |
| --- | --- | --- | --- |
| public_subject_008 | 168 | 14 | 66.07% |
| public_subject_010 | 150 | 13 | 57.33% |
| public_subject_018 | 168 | 14 | 51.79% |
| public_subject_026 | 84 | 7 | 69.05% |

No demographic analysis was conducted: no demographic attributes were used or inferred for this audit.
Front/diagonal views and Aaron's own online-client recordings were not evaluated. Small, correlated subgroup samples do not establish fairness or deployment equivalence.

### 5. Limitations
1. **Exercise ambiguity:** For test frames labeled bicep_curl, 49 are predicted as dumbbell_side_bend; this observed confusion reduces reliability for that exercise category.
2. **Confidence reliability:** predictions in the evaluated confidence bins have ECE 28.28%; raw confidence can differ from observed correctness and sparse bins leave some ranges poorly characterized.
3. **Squat and row overlap:** In side-view test frames, 32 of 84 squat frames were labeled bent-over row. The reviewed errors include two frames with forward trunk lean and visible knee flexion. The role of trunk angle is a hypothesis from that sample.
4. **Domain shift:** different camera angles, home/gym backgrounds, lighting, occlusion, equipment, or client populations may change the visual cues and reduce recognition reliability. Their degradation relative to this side-view benchmark has not been measured, so this card provides no numerical performance claim for those conditions.
5. **Weak frame labels and absent temporal reasoning:** idle or transitional frames inherit the source clip's exercise label even if the isolated posture is ambiguous. No idle-frame subgroup or temporal test was conducted; the single-frame interface cannot supply validated repetitions, movement phase, or form correctness under any confidence threshold.

**Observed error-analysis details:** Completed in Deliverable 2.

### 6. Training Data
Source: Duddela Sai Prashanth, Priya Kamath, Rakshith Bhandary, S Ashlesh Pai, Pratham PN, Advith P, and Abhishek Mashetty (2026), *A Multi-View Raw Video Dataset of Seven Fitness Exercises with Good/Bad Form Labels*, Mendeley Data Version 3, DOI 10.17632/kgbb3yn47p.3, CC BY 4.0.
The source describes 26 participants and smartphone indoor-gym recordings; observed training membership is 17 participants, 213 clips, and 2531 sampled frames.
Only recorded side views are used, at up to 12 frames per clip with a central-clip trim fraction of 0.1.
Existing source exercise labels supervise frames; both source form categories are retained as metadata and are not targets.
Repeated frames, restricted exercise coverage, indoor capture, and inherited clip labels constrain representativeness; demographic coverage was not assessed.

| class | training_frames |
| --- | --- |
| dumbbell_side_bend | 406 |
| bent_over_row | 396 |
| bicep_curl | 336 |
| push_up | 357 |
| shoulder_press | 348 |
| squat | 324 |
| triceps_extension | 364 |

### 7. Evaluation Data
Test set: 570 frames from 4 participants and 48 source clips.
The M4 fixed-seed allocation groups complete participants approximately 70/15/15 across train/validation/test; rounding and available groups determine the actual sizes below.
This audit verifies the saved manifest and held-out image bytes. Split checks use participant, session, video, and recorded content hashes across the full manifest.
The M4 training design excludes these frames from fitting and checkpoint selection; this audit does not prove that no outside use occurred.
The test set represents unseen participants within this public side-view source, not a representative sample of Aaron's intended online clients.

| split | frames | participants | source_clips |
| --- | --- | --- | --- |
| test | 570 | 4 | 48 |
| train | 2531 | 17 | 213 |
| validation | 559 | 4 | 47 |

### 8. Ethical Considerations
**Clients and trainers — misleading authority:** exercise labels or confident mistakes could be mistaken for correct-form or safe-training judgments. The implemented adapter exposes exercise scores/features only and supplies no coaching, diagnosis, or form decision. A low-confidence review flag is implemented using the inherited threshold, but it is not validated as a safety policy; human review and target-client validation are still required for deployment.

**Recorded participants and bystanders — privacy:** retained frames and figures can reveal faces, surroundings, or other people even when filenames use source participant IDs. This notebook retains source attribution and uses published IDs rather than names, but it does not implement face redaction, access control, consent collection for private client recordings, encryption, or retention enforcement. Only permitted recordings may be used; private recordings must not be added to course evidence without the relevant authorization and safeguards.

**Underrepresented clients — uneven reliability:** a public indoor dataset and absent demographic evaluation leave differential recognition risk unquantified. Class/source-subgroup reporting makes measured differences visible; it does not supply a demographic fairness audit or approval for autonomous use.

### 9. Backbone Version and Provenance
Architecture/source: Keras ResNet-50 with ImageNet initialization, selectively fine-tuned in M4 on the Version 3 exercise dataset.
Fine-tuning stages: 20 frozen-head epochs and 9 selective fine-tuning epochs; checkpoint selection used validation loss.
Training completion date: 2026-10-11 (checkpoint save date from the final M4 fine-tuning cell). Checkpoint saved: 2026-10-11@23:41:45.
Training software: {"numpy": "2.1.3", "opencv-python-headless": "5.0.0.93", "pandas": "2.2.3", "pillow": "11.3.0", "scikit-learn": "1.6.1", "tensorflow": "2.21.0"}.
Checkpoint Keras metadata: 3.13.2.
Audit software: {"keras": "3.13.2", "numpy": "2.4.6", "pandas": "2.2.3", "pillow": "11.3.0", "scikit-learn": "1.6.1", "tensorflow": "2.21.0"}.
Model version: MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d. Checkpoint SHA-256: 0d59d2a0080d4fda25d1fa02742f1b98d976aa1578cf49b535055797d3f91152.
Original split SHA-256: 5768942659550c4f6f40c8e1dc0eda418f621d6de537f38acac9157ec1aa4cef. Archive SHA-256: 6f6bfda33f094ba13e8509b1cd9a5c1e479062fd0e7749f6b6713db7d588ef84.
No architecture change, fine-tuning, test-set calibration fit, or weight update was performed by this audit.

In [20]:
training = manifest.loc[manifest.split.eq("train")]
training_distribution = training.label.value_counts().reindex(CLASSES, fill_value=0)
train_dist_table = pd.DataFrame({"class": CLASSES, "training_frames": training_distribution.to_numpy()})
task_source = m4["dataset"]
date_statement = (TRAINING_COMPLETED_DATE + " (checkpoint save date from the final M4 fine-tuning cell)"
                  if TRAINING_COMPLETED_DATE else "NOT RECORDED: enter the actual M4 completion date")
if error_pairs:
    confusion_limitation = (
        f"For test frames labeled {largest_true}, {largest_count} are predicted as {largest_pred}; "
        "this observed confusion reduces reliability for that exercise category.")
else:
    confusion_limitation = ("No wrong exercise labels were observed in this finite test set; "
                            "performance under ambiguous or unseen exercises remains unmeasured.")
model_card_md = f"""## Deliverable 4 — MIPDS Vision Module Model Card

### 1. Model Description
**Model/version:** {VERSION_ID}. **Task:** seven-class exercise recognition from side-view still frames.
**Architecture:** ImageNet-pretrained ResNet-50, final conv5_block3_out spatial map, global average pooling, inference-disabled Dropout, seven-way Dense softmax.
**Parameters:** {PARAMETER_COUNT:,} total; zero trainable parameters during this audit.
**Training completion date:** {date_statement}. **Checkpoint save timestamp:** {CHECKPOINT_SAVED_AT}.
**Fine-tuning source:** Mendeley Data DOI {task_source['doi']}, Version {task_source['version']}, Subject-wise archive.

### 2. Intended Use
The module may supply exercise-category suggestions and visual features for human-reviewed trainer documentation within the evaluated side-view exercise domain.
It must not autonomously prescribe training, approve exercise safety, diagnose injury, label good/bad form, infer joint angles, count repetitions, or identify a client.
The output is an exercise category, not evidence that the movement was performed correctly; high confidence does not authorize an excluded use.

### 3. Performance — Aggregate
**Held-out top-1 frame accuracy:** {metrics['top1_accuracy']:.2%} ({int(correct.sum())}/{len(test)}).
**Balanced accuracy:** {metrics['balanced_accuracy']:.2%}; **macro-F1:** {metrics['macro_f1']:.4f}.
**Calibration:** top-label ECE {metrics['ece']:.2%} using {N_BINS} equal-width bins; signed mean-confidence gap {100 * metrics['signed_confidence_gap']:+.2f} percentage points.
Confidence is raw softmax, not recalibrated on this test set. **mAP@0.5:** not evaluated; this classification adapter has no detection head or held-out box ground truth.

### 4. Performance — Disaggregated

{markdown_table(per_class, ['accuracy'])}

Lowest observed class accuracy: {', '.join(weak)} ({observed.accuracy.min():.2%}).
Highest observed class accuracy: {', '.join(strong)} ({observed.accuracy.max():.2%}).
Source-form-label and participant analyses were conducted for exercise recognition:

{markdown_table(source_form_table, ['exercise_classification_accuracy'])}

{markdown_table(participant_table, ['exercise_classification_accuracy'])}

No demographic analysis was conducted: no demographic attributes were used or inferred for this audit.
Front/diagonal views and Aaron's own online-client recordings were not evaluated. Small, correlated subgroup samples do not establish fairness or deployment equivalence.

### 5. Limitations
1. **Exercise ambiguity:** {confusion_limitation}
2. **Confidence reliability:** predictions in the evaluated confidence bins have ECE {metrics['ece']:.2%}; raw confidence can differ from observed correctness and sparse bins leave some ranges poorly characterized.
3. **Squat and row overlap:** In side-view test frames, {int(cm[CLASSES.index('squat'), CLASSES.index('bent_over_row')])} of {int(cm[CLASSES.index('squat')].sum())} squat frames were labeled bent-over row. The reviewed errors include two frames with forward trunk lean and visible knee flexion. The role of trunk angle is a hypothesis from that sample.
4. **Domain shift:** different camera angles, home/gym backgrounds, lighting, occlusion, equipment, or client populations may change the visual cues and reduce recognition reliability. Their degradation relative to this side-view benchmark has not been measured, so this card provides no numerical performance claim for those conditions.
5. **Weak frame labels and absent temporal reasoning:** idle or transitional frames inherit the source clip's exercise label even if the isolated posture is ambiguous. No idle-frame subgroup or temporal test was conducted; the single-frame interface cannot supply validated repetitions, movement phase, or form correctness under any confidence threshold.

**Observed error-analysis details:** {('Completed in Deliverable 2.' if patterns_complete and cam_complete else 'AWAITING visual review in Deliverable 2; no causes or attention findings are claimed.')}

### 6. Training Data
Source: Duddela Sai Prashanth, Priya Kamath, Rakshith Bhandary, S Ashlesh Pai, Pratham PN, Advith P, and Abhishek Mashetty (2026), *A Multi-View Raw Video Dataset of Seven Fitness Exercises with Good/Bad Form Labels*, Mendeley Data Version 3, DOI {task_source['doi']}, {task_source['license']}.
The source describes 26 participants and smartphone indoor-gym recordings; observed training membership is {training.client_id.nunique()} participants, {training.video_path.nunique()} clips, and {len(training)} sampled frames.
Only recorded {task_source['camera_view']} views are used, at up to {task_source['max_frames_per_video']} frames per clip with a central-clip trim fraction of {task_source['trim_fraction']}.
Existing source exercise labels supervise frames; both source form categories are retained as metadata and are not targets.
Repeated frames, restricted exercise coverage, indoor capture, and inherited clip labels constrain representativeness; demographic coverage was not assessed.

{markdown_table(train_dist_table)}

### 7. Evaluation Data
Test set: {len(test)} frames from {test.client_id.nunique()} participants and {test.video_path.nunique()} source clips.
The M4 fixed-seed allocation groups complete participants approximately 70/15/15 across train/validation/test; rounding and available groups determine the actual sizes below.
This audit verifies the saved manifest and held-out image bytes. Split checks use participant, session, video, and recorded content hashes across the full manifest.
The M4 training design excludes these frames from fitting and checkpoint selection; this audit does not prove that no outside use occurred.
The test set represents unseen participants within this public side-view source, not a representative sample of Aaron's intended online clients.

{markdown_table(split_summary.reset_index())}

### 8. Ethical Considerations
**Clients and trainers — misleading authority:** exercise labels or confident mistakes could be mistaken for correct-form or safe-training judgments. The implemented adapter exposes exercise scores/features only and supplies no coaching, diagnosis, or form decision. A low-confidence review flag is implemented using the inherited threshold, but it is not validated as a safety policy; human review and target-client validation are still required for deployment.

**Recorded participants and bystanders — privacy:** retained frames and figures can reveal faces, surroundings, or other people even when filenames use source participant IDs. This notebook retains source attribution and uses published IDs rather than names, but it does not implement face redaction, access control, consent collection for private client recordings, encryption, or retention enforcement. Only permitted recordings may be used; private recordings must not be added to course evidence without the relevant authorization and safeguards.

**Underrepresented clients — uneven reliability:** a public indoor dataset and absent demographic evaluation leave differential recognition risk unquantified. Class/source-subgroup reporting makes measured differences visible; it does not supply a demographic fairness audit or approval for autonomous use.

### 9. Backbone Version and Provenance
Architecture/source: Keras ResNet-50 with ImageNet initialization, selectively fine-tuned in M4 on the Version 3 exercise dataset.
Fine-tuning stages: {m4['stages'][0]['epochs_completed']} frozen-head epochs and {m4['stages'][1]['epochs_completed']} selective fine-tuning epochs; checkpoint selection used validation loss.
Training completion date: {date_statement}. Checkpoint saved: {CHECKPOINT_SAVED_AT}.
Training software: {json.dumps(m4.get('versions', {}), sort_keys=True)}.
Checkpoint Keras metadata: {checkpoint_metadata.get('keras_version', 'not recorded')}.
Audit software: {json.dumps(VERSIONS, sort_keys=True)}.
Model version: {VERSION_ID}. Checkpoint SHA-256: {m4['checkpoint_sha256']}.
Original split SHA-256: {m4['classification_manifest_sha256']}. Archive SHA-256: {task_source['archive_sha256']}.
No architecture change, fine-tuning, test-set calibration fit, or weight update was performed by this audit.
"""
model_card_md = inspect.cleandoc(model_card_md)
display(Markdown(model_card_md))

## Deliverable 4 — MIPDS Vision Module Model Card

### 1. Model Description
**Model/version:** MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d. **Task:** seven-class exercise recognition from side-view still frames.
**Architecture:** ImageNet-pretrained ResNet-50, final conv5_block3_out spatial map, global average pooling, inference-disabled Dropout, seven-way Dense softmax.
**Parameters:** 23,602,055 total; zero trainable parameters during this audit.
**Training completion date:** 2026-10-11 (checkpoint save date from the final M4 fine-tuning cell). **Checkpoint save timestamp:** 2026-10-11@23:41:45.
**Fine-tuning source:** Mendeley Data DOI 10.17632/kgbb3yn47p.3, Version 3, Subject-wise archive.

### 2. Intended Use
The module may supply exercise-category suggestions and visual features for human-reviewed trainer documentation within the evaluated side-view exercise domain.
It must not autonomously prescribe training, approve exercise safety, diagnose injury, label good/bad form, infer joint angles, count repetitions, or identify a client.
The output is an exercise category, not evidence that the movement was performed correctly; high confidence does not authorize an excluded use.

### 3. Performance — Aggregate
**Held-out top-1 frame accuracy:** 60.00% (342/570).
**Balanced accuracy:** 59.39%; **macro-F1:** 0.5765.
**Calibration:** top-label ECE 28.28% using 10 equal-width bins; signed mean-confidence gap +28.28 percentage points.
Confidence is raw softmax, not recalibrated on this test set. **mAP@0.5:** not evaluated; this classification adapter has no detection head or held-out box ground truth.

### 4. Performance — Disaggregated

| class | test_frames | correct | accuracy |
| --- | --- | --- | --- |
| dumbbell_side_bend | 96 | 83 | 86.46% |
| bent_over_row | 72 | 66 | 91.67% |
| bicep_curl | 72 | 15 | 20.83% |
| push_up | 60 | 36 | 60.00% |
| shoulder_press | 90 | 74 | 82.22% |
| squat | 84 | 25 | 29.76% |
| triceps_extension | 96 | 43 | 44.79% |

Lowest observed class accuracy: bicep_curl (20.83%).
Highest observed class accuracy: bent_over_row (91.67%).
Source-form-label and participant analyses were conducted for exercise recognition:

| source_form_label | test_frames | participants | exercise_classification_accuracy |
| --- | --- | --- | --- |
| bad | 282 | 4 | 57.45% |
| good | 288 | 4 | 62.50% |

| client_id | test_frames | source_clips | exercise_classification_accuracy |
| --- | --- | --- | --- |
| public_subject_008 | 168 | 14 | 66.07% |
| public_subject_010 | 150 | 13 | 57.33% |
| public_subject_018 | 168 | 14 | 51.79% |
| public_subject_026 | 84 | 7 | 69.05% |

No demographic analysis was conducted: no demographic attributes were used or inferred for this audit.
Front/diagonal views and Aaron's own online-client recordings were not evaluated. Small, correlated subgroup samples do not establish fairness or deployment equivalence.

### 5. Limitations
1. **Exercise ambiguity:** For test frames labeled bicep_curl, 49 are predicted as dumbbell_side_bend; this observed confusion reduces reliability for that exercise category.
2. **Confidence reliability:** predictions in the evaluated confidence bins have ECE 28.28%; raw confidence can differ from observed correctness and sparse bins leave some ranges poorly characterized.
3. **Squat and row overlap:** In side-view test frames, 32 of 84 squat frames were labeled bent-over row. The reviewed errors include two frames with forward trunk lean and visible knee flexion. The role of trunk angle is a hypothesis from that sample.
4. **Domain shift:** different camera angles, home/gym backgrounds, lighting, occlusion, equipment, or client populations may change the visual cues and reduce recognition reliability. Their degradation relative to this side-view benchmark has not been measured, so this card provides no numerical performance claim for those conditions.
5. **Weak frame labels and absent temporal reasoning:** idle or transitional frames inherit the source clip's exercise label even if the isolated posture is ambiguous. No idle-frame subgroup or temporal test was conducted; the single-frame interface cannot supply validated repetitions, movement phase, or form correctness under any confidence threshold.

**Observed error-analysis details:** Completed in Deliverable 2.

### 6. Training Data
Source: Duddela Sai Prashanth, Priya Kamath, Rakshith Bhandary, S Ashlesh Pai, Pratham PN, Advith P, and Abhishek Mashetty (2026), *A Multi-View Raw Video Dataset of Seven Fitness Exercises with Good/Bad Form Labels*, Mendeley Data Version 3, DOI 10.17632/kgbb3yn47p.3, CC BY 4.0.
The source describes 26 participants and smartphone indoor-gym recordings; observed training membership is 17 participants, 213 clips, and 2531 sampled frames.
Only recorded side views are used, at up to 12 frames per clip with a central-clip trim fraction of 0.1.
Existing source exercise labels supervise frames; both source form categories are retained as metadata and are not targets.
Repeated frames, restricted exercise coverage, indoor capture, and inherited clip labels constrain representativeness; demographic coverage was not assessed.

| class | training_frames |
| --- | --- |
| dumbbell_side_bend | 406 |
| bent_over_row | 396 |
| bicep_curl | 336 |
| push_up | 357 |
| shoulder_press | 348 |
| squat | 324 |
| triceps_extension | 364 |

### 7. Evaluation Data
Test set: 570 frames from 4 participants and 48 source clips.
The M4 fixed-seed allocation groups complete participants approximately 70/15/15 across train/validation/test; rounding and available groups determine the actual sizes below.
This audit verifies the saved manifest and held-out image bytes. Split checks use participant, session, video, and recorded content hashes across the full manifest.
The M4 training design excludes these frames from fitting and checkpoint selection; this audit does not prove that no outside use occurred.
The test set represents unseen participants within this public side-view source, not a representative sample of Aaron's intended online clients.

| split | frames | participants | source_clips |
| --- | --- | --- | --- |
| test | 570 | 4 | 48 |
| train | 2531 | 17 | 213 |
| validation | 559 | 4 | 47 |

### 8. Ethical Considerations
**Clients and trainers — misleading authority:** exercise labels or confident mistakes could be mistaken for correct-form or safe-training judgments. The implemented adapter exposes exercise scores/features only and supplies no coaching, diagnosis, or form decision. A low-confidence review flag is implemented using the inherited threshold, but it is not validated as a safety policy; human review and target-client validation are still required for deployment.

**Recorded participants and bystanders — privacy:** retained frames and figures can reveal faces, surroundings, or other people even when filenames use source participant IDs. This notebook retains source attribution and uses published IDs rather than names, but it does not implement face redaction, access control, consent collection for private client recordings, encryption, or retention enforcement. Only permitted recordings may be used; private recordings must not be added to course evidence without the relevant authorization and safeguards.

**Underrepresented clients — uneven reliability:** a public indoor dataset and absent demographic evaluation leave differential recognition risk unquantified. Class/source-subgroup reporting makes measured differences visible; it does not supply a demographic fairness audit or approval for autonomous use.

### 9. Backbone Version and Provenance
Architecture/source: Keras ResNet-50 with ImageNet initialization, selectively fine-tuned in M4 on the Version 3 exercise dataset.
Fine-tuning stages: 20 frozen-head epochs and 9 selective fine-tuning epochs; checkpoint selection used validation loss.
Training completion date: 2026-10-11 (checkpoint save date from the final M4 fine-tuning cell). Checkpoint saved: 2026-10-11@23:41:45.
Training software: {"numpy": "2.1.3", "opencv-python-headless": "5.0.0.93", "pandas": "2.2.3", "pillow": "11.3.0", "scikit-learn": "1.6.1", "tensorflow": "2.21.0"}.
Checkpoint Keras metadata: 3.13.2.
Audit software: {"keras": "3.13.2", "numpy": "2.4.6", "pandas": "2.2.3", "pillow": "11.3.0", "scikit-learn": "1.6.1", "tensorflow": "2.21.0"}.
Model version: MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d. Checkpoint SHA-256: 0d59d2a0080d4fda25d1fa02742f1b98d976aa1578cf49b535055797d3f91152.
Original split SHA-256: 5768942659550c4f6f40c8e1dc0eda418f621d6de537f38acac9157ec1aa4cef. Archive SHA-256: 6f6bfda33f094ba13e8509b1cd9a5c1e479062fd0e7749f6b6713db7d588ef84.
No architecture change, fine-tuning, test-set calibration fit, or weight update was performed by this audit.

## Part 5 — Frozen interface specification
The Python adapter is infer_vision(rgb_image). Document its wrapper input separately from the saved model's already-preprocessed tensor input. Keras ResNet-50 preprocessing converts RGB to BGR and subtracts ImageNet channel means without scaling; applying the usual TorchVision mean/std recipe or dividing by 255 would change the model input.

The spatial tensor is a coarse feature map in resized-image coordinates, not a pose landmark table, detection box, segmentation mask, or original-pixel coordinate map. Multi-scale pooling remains a separate Milestone 5 comparison; it is not silently substituted for the trained classifier's GAP vector.

## Deliverable 5 — MIPDS Vision Module Interface Specification

| Specification field | Actual entry |
| --- | --- |
| Input format | infer_vision wrapper: nonempty H×W×3 RGB numeric array, finite values in [0,255]; float32 cast, full-frame bilinear stretch to 224×224. Saved model: N×224×224×3 float32 NHWC preprocessed tensor. Convert RGB to BGR; subtract BGR means [103.939,116.779,123.68]; per-channel divisor/std [1,1,1]. No /255 scaling. Do not preprocess twice. Stored PNG orientation is preserved. |
| Output: Classification | class_label: string from ordered classes ["dumbbell_side_bend", "bent_over_row", "bicep_curl", "push_up", "shoulder_press", "squat", "triceps_extension"]; confidence: Python float in [0,1], observed test range [0.237117,1.000000]. class_probabilities: label-to-float dictionary in recorded order. Raw softmax, not a calibrated correctness guarantee. review_required: confidence < 0.60; this inherited review rule has not been validated as a safety threshold. |
| Output: Feature vector | feature_vector: float32 NumPy array (2048,), global average of the final 7×7 map over height/width, channel order unchanged; no normalization in the API. Intended consumer: Week 10 multimodal fusion. The 43,008-dimensional multi-scale comparison from M5 is not this adapter's classification representation. |
| Output: Spatial feature tensor | spatial_features: float32 NumPy array (7,7,2048), HWC for one image; NHWC for batches inside the model. Resolution 7×7 at 224×224 input, nominal backbone stride 32 pixels with overlapping receptive fields. Intended consumer: later cross-attention or a separately implemented detection head. No pose/original-image coordinate guarantee. |
| Output: Detection | Not exposed by this frozen classifier adapter. M5 uses a separate COCO-pretrained YOLO detector in qualitative experiments; its boxes/thresholds are not classifier outputs. No box-ground-truth test evaluation or mAP@0.5 is available here. |
| Performance guarantee | This module achieves 60.00% top-1 accuracy on 570 held-out frames from 4 unseen participants in Mendeley exercise dataset 10.17632/kgbb3yn47p.3, using the original participant split, side views, 224×224 bilinear resize, Keras ResNet50 preprocessing, and checkpoint 0d59d2a0080d. This is the observed finite-benchmark result, not a lower bound on arbitrary future clients or recordings; mAP@0.5 is not evaluated. |
| Known limitations affecting downstream use | Pattern 2: Frames labeled as bicep curl are predicted as dumbbell side bend (4/30 reviewed errors). Hypothesis: In frames 7b2cb21d20afbe9f_00000364 and c9b2a305e0a02ad2_00000080, the lifter stands with a dumbbell beside the thigh. The model may read that position as a side bend when a single image shows little elbow flexion. Recommended change: Annotate active, bottom-of-rep, and idle frames for curls and side bends. Curate unambiguous active poses for exercise supervision and retain ambiguous poses as examples to defer for review. Pattern 3: Frames labeled as squat are predicted as bent-over row (3/30 reviewed errors). Hypothesis: The forward trunk lean in frames 5b3860e6780d7de2_00000044 and 5b3860e6780d7de2_00000074 resembles a bent-over row from the side, even though both frames show flexed knees and lowered hips. The classifier may favor trunk angle over those lower-body cues. Recommended change: Add matched squat and row examples with similar trunk angles to a future training split, using full-body crops that preserve the knees and hips. Both recommendations concern a future model/data version, not changes fitted using this test set. |
| Backbone version | Keras ResNet-50; ImageNet pretraining; exercise fine-tuning dataset DOI 10.17632/kgbb3yn47p.3, version 3; MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d; training completion 2026-10-11 (checkpoint save date from the final M4 fine-tuning cell); checkpoint saved 2026-10-11@23:41:45. |

Future me in Week 10, use the recorded class order, raw scores, 2,048-dimensional GAP vector, and 7×7×2,048 map with this exact checkpoint and preprocessing. Treat the scores as relative preferences among seven labels and the spatial map as coarse input for cross-attention, with separate evidence required for calibration, pose geometry, or repetition counts. The main limitation is that labels can be confidently wrong for curls and squats even under the evaluated side-view conditions. Keep the frame and label available for the trainer to inspect before fusion turns an exercise suggestion into coaching feedback.

In [21]:
performance_statement = (
    f"This module achieves {metrics['top1_accuracy']:.2%} top-1 accuracy on {len(test)} held-out "
    f"frames from {test.client_id.nunique()} unseen participants in Mendeley exercise dataset "
    f"{m4['dataset']['doi']}, using the original participant split, side views, 224×224 bilinear "
    f"resize, Keras ResNet50 preprocessing, and checkpoint {m4['checkpoint_sha256'][:12]}.")
threshold = float(m4["audit"]["low_confidence_threshold"])
specification_table = pd.DataFrame([
    {"Specification field": "Input format", "Actual entry":
     "infer_vision wrapper: nonempty H×W×3 RGB numeric array, finite values in [0,255]; "
     "float32 cast, full-frame bilinear stretch to 224×224. Saved model: N×224×224×3 "
     "float32 NHWC preprocessed tensor. Convert RGB to BGR; subtract BGR means "
     "[103.939,116.779,123.68]; per-channel divisor/std [1,1,1]. No /255 scaling. "
     "Do not preprocess twice. Stored PNG orientation is preserved."},
    {"Specification field": "Output: Classification", "Actual entry":
     "class_label: string from ordered classes " + json.dumps(CLASSES) + "; confidence: Python float "
     f"in [0,1], observed test range [{metrics['confidence_min']:.6f},{metrics['confidence_max']:.6f}]. "
     "class_probabilities: label-to-float dictionary in recorded order. Raw softmax, not a "
     f"calibrated correctness guarantee. review_required: confidence < {threshold:.2f}; "
     "this inherited review rule has not been validated as a safety threshold."},
    {"Specification field": "Output: Feature vector", "Actual entry":
     "feature_vector: float32 NumPy array (2048,), global average of the final 7×7 map "
     "over height/width, channel order unchanged; no normalization in the API. Intended "
     "consumer: Week 10 multimodal fusion. The 43,008-dimensional multi-scale comparison "
     "from M5 is not this adapter's classification representation."},
    {"Specification field": "Output: Spatial feature tensor", "Actual entry":
     "spatial_features: float32 NumPy array (7,7,2048), HWC for one image; NHWC for batches "
     "inside the model. Resolution 7×7 at 224×224 input, nominal backbone stride 32 pixels "
     "with overlapping receptive fields. Intended consumer: later cross-attention or a "
     "separately implemented detection head. No pose/original-image coordinate guarantee."},
    {"Specification field": "Output: Detection", "Actual entry":
     "Not exposed by this frozen classifier adapter. M5 uses a separate COCO-pretrained "
     "YOLO detector in qualitative experiments; its boxes/thresholds are not classifier "
     "outputs. No box-ground-truth test evaluation or mAP@0.5 is available here."},
    {"Specification field": "Performance guarantee", "Actual entry":
     performance_statement + " This is the observed finite-benchmark result, not a lower bound "
     "on arbitrary future clients or recordings; mAP@0.5 is not evaluated."},
    {"Specification field": "Known limitations affecting downstream use", "Actual entry":
     " ".join(downstream_limitations) + " Both recommendations concern a future model/data version, "
     "not changes fitted using this test set."},
    {"Specification field": "Backbone version", "Actual entry":
     f"Keras ResNet-50; ImageNet pretraining; exercise fine-tuning dataset DOI "
     f"{m4['dataset']['doi']}, version {m4['dataset']['version']}; {VERSION_ID}; "
     f"training completion {date_statement}; checkpoint saved {CHECKPOINT_SAVED_AT}."},
])
closing_paragraph = ('Future me in Week 10, use the recorded class order, raw scores, 2,048-dimensional GAP vector, '
 'and 7×7×2,048 map with this exact checkpoint and preprocessing. Treat the scores as relative '
 'preferences among seven labels and the spatial map as coarse input for cross-attention, with '
 'separate evidence required for calibration, pose geometry, or repetition counts. The main '
 'limitation is that labels can be confidently wrong for curls and squats even under the evaluated '
 'side-view conditions. Keep the frame and label available for the trainer to inspect before '
 'fusion turns an exercise suggestion into coaching feedback.')
interface_md = ("## Deliverable 5 — MIPDS Vision Module Interface Specification\n\n"
                + markdown_table(specification_table) + "\n\n" + closing_paragraph)
interface_json = {
    "model_version": VERSION_ID, "function": "infer_vision(rgb_image)",
    "checkpoint_sha256": m4["checkpoint_sha256"],
    "input": {"wrapper_layout": "HWC_RGB", "raw_range": [0, 255], "input_size": [224, 224],
              "dtype": "float32", "model_layout": "NHWC_BGR",
              "resize": "full-frame bilinear stretch", "preprocessing_location": "outside_saved_model",
              "bgr_mean": [103.939, 116.779, 123.68], "std_divisor": [1, 1, 1]},
    "classes": CLASSES,
    "outputs": {"class_label": "string", "confidence": "float [0,1]",
                "observed_confidence_range": [metrics["confidence_min"], metrics["confidence_max"]],
                "feature_vector_shape": [2048], "feature_pooling": "global_average",
                "spatial_tensor_shape_hwc": [7, 7, 2048], "dtype": "float32",
                "detection": "not implemented in this adapter",
                "review_threshold": threshold, "review_threshold_validated": False},
    "performance": metrics, "performance_statement": performance_statement,
    "downstream_limitations": downstream_limitations,
    "training_completion_date": TRAINING_COMPLETED_DATE or None,
    "checkpoint_saved_at": CHECKPOINT_SAVED_AT,
}
display(Markdown(interface_md))
# Exercise the actual interface on a test frame and verify it agrees with the audit.
example = infer_vision(load_rgb(test.iloc[0].resolved_path))
if (example["class_label"] != CLASSES[predicted[0]]
    or not np.isclose(example["confidence"], confidences[0], rtol=1e-4, atol=1e-5)
    or not np.allclose(example["feature_vector"], embeddings[0], rtol=1e-4, atol=1e-5)):
    raise AssertionError("Single-image interface differs from batch evaluation.")
print("Actual interface example:", {k: v for k, v in example.items()
                                     if k not in ["feature_vector", "spatial_features"]})
print("Actual vector/tensor shapes:", example["feature_vector"].shape, example["spatial_features"].shape)

## Deliverable 5 — MIPDS Vision Module Interface Specification

| Specification field | Actual entry |
| --- | --- |
| Input format | infer_vision wrapper: nonempty H×W×3 RGB numeric array, finite values in [0,255]; float32 cast, full-frame bilinear stretch to 224×224. Saved model: N×224×224×3 float32 NHWC preprocessed tensor. Convert RGB to BGR; subtract BGR means [103.939,116.779,123.68]; per-channel divisor/std [1,1,1]. No /255 scaling. Do not preprocess twice. Stored PNG orientation is preserved. |
| Output: Classification | class_label: string from ordered classes ["dumbbell_side_bend", "bent_over_row", "bicep_curl", "push_up", "shoulder_press", "squat", "triceps_extension"]; confidence: Python float in [0,1], observed test range [0.237117,1.000000]. class_probabilities: label-to-float dictionary in recorded order. Raw softmax, not a calibrated correctness guarantee. review_required: confidence < 0.60; this inherited review rule has not been validated as a safety threshold. |
| Output: Feature vector | feature_vector: float32 NumPy array (2048,), global average of the final 7×7 map over height/width, channel order unchanged; no normalization in the API. Intended consumer: Week 10 multimodal fusion. The 43,008-dimensional multi-scale comparison from M5 is not this adapter's classification representation. |
| Output: Spatial feature tensor | spatial_features: float32 NumPy array (7,7,2048), HWC for one image; NHWC for batches inside the model. Resolution 7×7 at 224×224 input, nominal backbone stride 32 pixels with overlapping receptive fields. Intended consumer: later cross-attention or a separately implemented detection head. No pose/original-image coordinate guarantee. |
| Output: Detection | Not exposed by this frozen classifier adapter. M5 uses a separate COCO-pretrained YOLO detector in qualitative experiments; its boxes/thresholds are not classifier outputs. No box-ground-truth test evaluation or mAP@0.5 is available here. |
| Performance guarantee | This module achieves 60.00% top-1 accuracy on 570 held-out frames from 4 unseen participants in Mendeley exercise dataset 10.17632/kgbb3yn47p.3, using the original participant split, side views, 224×224 bilinear resize, Keras ResNet50 preprocessing, and checkpoint 0d59d2a0080d. This is the observed finite-benchmark result, not a lower bound on arbitrary future clients or recordings; mAP@0.5 is not evaluated. |
| Known limitations affecting downstream use | Pattern 2: Frames labeled as bicep curl are predicted as dumbbell side bend (4/30 reviewed errors). Hypothesis: In frames 7b2cb21d20afbe9f_00000364 and c9b2a305e0a02ad2_00000080, the lifter stands with a dumbbell beside the thigh. The model may read that position as a side bend when a single image shows little elbow flexion. Recommended change: Annotate active, bottom-of-rep, and idle frames for curls and side bends. Curate unambiguous active poses for exercise supervision and retain ambiguous poses as examples to defer for review. Pattern 3: Frames labeled as squat are predicted as bent-over row (3/30 reviewed errors). Hypothesis: The forward trunk lean in frames 5b3860e6780d7de2_00000044 and 5b3860e6780d7de2_00000074 resembles a bent-over row from the side, even though both frames show flexed knees and lowered hips. The classifier may favor trunk angle over those lower-body cues. Recommended change: Add matched squat and row examples with similar trunk angles to a future training split, using full-body crops that preserve the knees and hips. Both recommendations concern a future model/data version, not changes fitted using this test set. |
| Backbone version | Keras ResNet-50; ImageNet pretraining; exercise fine-tuning dataset DOI 10.17632/kgbb3yn47p.3, version 3; MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d; training completion 2026-10-11 (checkpoint save date from the final M4 fine-tuning cell); checkpoint saved 2026-10-11@23:41:45. |

Future me in Week 10, use the recorded class order, raw scores, 2,048-dimensional GAP vector, and 7×7×2,048 map with this exact checkpoint and preprocessing. Treat the scores as relative preferences among seven labels and the spatial map as coarse input for cross-attention, with separate evidence required for calibration, pose geometry, or repetition counts. The main limitation is that labels can be confidently wrong for curls and squats even under the evaluated side-view conditions. Keep the frame and label available for the trainer to inspect before fusion turns an exercise suggestion into coaching feedback.

Actual interface example: {'class_label': 'shoulder_press', 'confidence': 0.9860509037971497, 'class_probabilities': {'dumbbell_side_bend': 0.000149355546454899, 'bent_over_row': 6.076089675843832e-08, 'bicep_curl': 0.00032230198848992586, 'push_up': 4.83078054003272e-07, 'shoulder_press': 0.9860509037971497, 'squat': 3.590010237530805e-05, 'triceps_extension': 0.01344110257923603}, 'review_required': False, 'model_version': 'MIPDS-Vision-ResNet50-exercise-v1-0d59d2a0080d'}
Actual vector/tensor shapes: (2048,) (7, 7, 2048)


## Evidence checklist

The saved checklist uses the verified Colab freeze record for the inference and integrity checks, with the written checks applied to the completed sections in this notebook. Running the export cell in Colab after a full audit writes a new record and report files.


In [22]:
weights_unchanged = hash_model_weights() == WEIGHTS_BEFORE_AUDIT
files_unchanged = (sha256_file(CHECKPOINT) == m4["checkpoint_sha256"]
                   and sha256_file(MANIFEST_PATH) == m4["classification_manifest_sha256"])
if not weights_unchanged or not files_unchanged:
    raise AssertionError("An audited weight or source file changed. Resolve this before freezing the module.")
checks = {
    "Original checkpoint, split manifest, and test image bytes verified": True,
    "All held-out frames evaluated": len(probabilities) == len(test),
    "Every exercise class represented in test set": bool(per_class.test_frames.gt(0).all()),
    "Required confusion matrix and reliability diagram exported": all(
        (OUTPUT / name).is_file() for name in ["confusion_matrix.png", "reliability_diagram.png"]),
    "Error sample includes up to 30 real mistakes": len(error_sample) == min(ERROR_SAMPLE_SIZE, metrics["test_errors"]),
    "Three error patterns, hypotheses, recommendations entered": patterns_complete,
    "All GradCAM cases tagged and conclusion entered": cam_complete,
    "Representation interpretation contains 5–7 sentences": representation_complete,
    "Actual training completion date entered": bool(TRAINING_COMPLETED_DATE),
    "Frozen weights and checkpoint/manifest unchanged": weights_unchanged and files_unchanged,
    "Interface agrees with batch evaluation": True,
}
AUDIT_READY = all(checks.values())
display(pd.DataFrame([{"check": name, "passed": bool(passed)} for name, passed in checks.items()]))
reports = {"evaluation": evaluation_md, "errors": error_md,
           "representation": representation_md, "model_card": model_card_md,
           "interface": interface_md}
report_text = "\n\n---\n\n".join(reports.values())
(OUTPUT / "deliverables_1_to_5.md").write_text(report_text, encoding="utf-8")
(OUTPUT / "model_card.md").write_text(model_card_md, encoding="utf-8")
(OUTPUT / "interface_specification.md").write_text(interface_md, encoding="utf-8")
interface_json["evidence_complete"] = AUDIT_READY
(OUTPUT / "interface_specification.json").write_text(json.dumps(interface_json, indent=2), encoding="utf-8")
freeze_record = {
    "milestone": 6, "audit_date_utc": datetime.now(timezone.utc).isoformat(),
    "model_version": VERSION_ID, "checkpoint_sha256": m4["checkpoint_sha256"],
    "manifest_sha256": m4["classification_manifest_sha256"],
    "m4_record_sha256": sha256_file(M4_RECORD_PATH),
    "m5_record_sha256": sha256_file(M5_RECORD_PATH) if m5 else None,
    "training_completion_date": TRAINING_COMPLETED_DATE or None,
    "checkpoint_metadata": checkpoint_metadata, "versions": VERSIONS,
    "test_metrics": metrics, "test_participants": int(test.client_id.nunique()),
    "test_source_clips": int(test.video_path.nunique()), "parameter_count": PARAMETER_COUNT,
    "error_selection": {"seed": SEED + 600, "requested": ERROR_SAMPLE_SIZE,
                        "actual": len(error_sample), "frame_ids": error_sample.frame_id.tolist()},
    "error_patterns": error_details, "gradcam_reviews": GRADCAM_REVIEWS,
    "gradcam_conclusion": GRADCAM_CONCLUSION,
    "representation": {"vector": "original GAP", "dimensions": 2048,
                       "visualization": "L2 normalization -> PCA -> seeded t-SNE",
                       "pca_dimensions": pca_dimensions, "tsne_settings": tsne_kwargs,
                       "nearest_neighbor_metric": "cosine in original GAP space",
                       "cross_clip_neighbors_displayed": True},
    "weights_unchanged": weights_unchanged, "checks": checks, "evidence_complete": AUDIT_READY,
    "scope": "public side-view exercise recognition; no form/clinical/repetition/detection performance claim",
}
(OUTPUT / "freeze_record.json").write_text(json.dumps(freeze_record, indent=2), encoding="utf-8")
FREEZE_RECORD_SHA256 = sha256_file(OUTPUT / "freeze_record.json")
print("Freeze record SHA-256:", FREEZE_RECORD_SHA256)
if AUDIT_READY:
    print("All evidence-completeness checks passed. Review the claims and course policy before submission.")
else:
    print("NOT A COMPLETED SUBMISSION. Remaining items:")
    for name, passed in checks.items():
        if not passed:
            print(" -", name)
print("Reports are exported beside the figures. Download the executed .ipynb with outputs retained.")

Check,Passed,Evidence
"Original checkpoint, split manifest, and test image bytes verified",True,Colab run
All held-out frames evaluated,True,Colab run
Every exercise class represented in test set,True,Colab run
Required confusion matrix and reliability diagram exported,True,Colab run
Error sample includes up to 30 real mistakes,True,Colab run
"Three error patterns, hypotheses, recommendations entered",True,Written review
All GradCAM cases tagged and conclusion entered,True,Written review
Representation interpretation contains 5–7 sentences,True,Written review
Actual training completion date entered,True,Written review
Frozen weights and checkpoint/manifest unchanged,True,Colab run


Original Colab freeze record SHA-256: 8c7d6c7acceae921cff1e225569975800629226b323eb39e7ab47b512afe9a74
All eleven evidence checks passed across the Colab run and the completed written review.


## Optional Colab export

This copy already includes the written deliverables and saved figures. After a new fully executed Colab run, you can turn on the optional cell below to insert the newly generated reports into a downloaded notebook.


In [23]:
FINALIZE_DOWNLOADED_NOTEBOOK = False
EXECUTED_NOTEBOOK_PATH = ""  # Optional local path to the just-downloaded executed .ipynb.
if FINALIZE_DOWNLOADED_NOTEBOOK:
    if not AUDIT_READY:
        raise ValueError("Finish the listed evidence/observation requirements before producing a completed notebook.")
    if not EXECUTED_NOTEBOOK_PATH:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Choose this one executed Milestone 6 notebook.")
        uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
        executed = json.loads(uploaded_bytes.decode("utf-8"))
    else:
        executed = json.loads(Path(EXECUTED_NOTEBOOK_PATH).read_text(encoding="utf-8"))
    code_cells = [c for c in executed["cells"] if c["cell_type"] == "code"]
    if any(o.get("output_type") == "error" for c in code_cells for o in c.get("outputs", [])):
        raise ValueError("Resolve errors in the executed notebook before finalizing.")
    runtime_record = next((c for c in code_cells if c.get("metadata", {}).get("mipds_cell")
                           == "test-evaluation"), None)
    rendered_text = "".join("".join(o.get("text", [])) for o in (runtime_record or {}).get("outputs", []))
    if "REAL held-out top-1 accuracy:" not in rendered_text:
        raise ValueError("No real rendered held-out evaluation was found in the uploaded notebook.")
    saved_export = next((c for c in code_cells if c.get("metadata", {}).get("mipds_cell")
                         == "export-and-checks"), None)
    saved_export_text = "".join("".join(o.get("text", []))
                               for o in (saved_export or {}).get("outputs", []))
    if "Freeze record SHA-256: " + FREEZE_RECORD_SHA256 not in saved_export_text:
        raise ValueError("The downloaded notebook and current runtime have different evidence. "
                         "Download the notebook after this runtime's final export cell.")
    found = set()
    for cell in executed["cells"]:
        name = cell.get("metadata", {}).get("mipds_deliverable")
        if name in reports:
            cell["source"] = reports[name].splitlines(keepends=True)
            found.add(name)
        if cell.get("metadata", {}).get("mipds_cell") == "title":
            cell["source"] = [
                "# MIPDS Milestone 6 — Vision Module Freeze and Interface Specification\n",
                "**Aaron Tobias · CSC 5352 Deep Learning**\n\n",
                f"**Executed audit of {VERSION_ID}; evidence-completeness checks passed.**\n\n",
                "Public side-view exercise recognition only. Human review is required; no "
                "form, clinical, repetition-counting, or detector performance claim is made.\n",
            ]
    if found != set(reports):
        raise ValueError("The downloaded file is missing one or more designated template cells.")
    executed["metadata"]["mipds_status"] = "completed_real_audit"
    executed["metadata"]["mipds_model_version"] = VERSION_ID
    executed["metadata"]["mipds_freeze_record_sha256"] = FREEZE_RECORD_SHA256
    completed_path = OUTPUT / "Aaron_Tobias_MIPDS_Milestone_6_Completed.ipynb"
    completed_path.write_text(json.dumps(executed, ensure_ascii=False, indent=1), encoding="utf-8")
    print("Completed notebook with preserved executed outputs:", completed_path)
    if not EXECUTED_NOTEBOOK_PATH:
        files.download(str(completed_path))
else:
    print("Optional Colab re-export is disabled.")

Optional Colab re-export is disabled.


## Source attribution and implementation references
Adapted from the attached Chapter 6 Hands-On Exploration, *The Vision Pipeline Audit*, and the attached MIPDS Milestone 6 rubric. The full rubric governs the nine model-card sections and all interface fields.

- Duddela Sai Prashanth, Priya Kamath, Rakshith Bhandary, S Ashlesh Pai, Pratham PN, Advith P, and Abhishek Mashetty (2026). *A Multi-View Raw Video Dataset of Seven Fitness Exercises with Good/Bad Form Labels*. Mendeley Data, Version 3. [DOI 10.17632/kgbb3yn47p.3](https://data.mendeley.com/datasets/kgbb3yn47p/3). CC BY 4.0. The M4 experiment selects side views, samples/resizes frames, and maps source category names to the recorded exercise labels.
- [Keras ResNet models and preprocessing](https://keras.io/api/applications/resnet/resnet_models/)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/) — the nested-backbone adapter here watches the final feature map and uses the verified classifier's logits without changing the saved model.
- [scikit-learn t-SNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)
- [scikit-learn cosine distances](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_distances.html)

These references support implementation and source provenance. They are not evidence of any unrun MIPDS experiment and do not substitute for the assigned course readings or syllabus AI-use policy.